# Face Recognition & Neural Style Transfer

## ✍️ Toy Examples

These face-recognition and style-transfer toys isolate the numeric mechanics with tiny embeddings and feature maps. Each block prints the intermediate distances, losses, or matrices, pins the result with an assertion, and draws one picture.

### ✍️ Toy 1 · Embedding distance, verification threshold, and recognition

Verification compares one claimed identity against a threshold; recognition compares the query to the whole gallery and chooses the nearest embedding.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)  # -> reproducible generator seeded with 0
t1_names = np.array(["Ava", "Ben", "Cy", "Dee"])  # -> four enrolled identities
t1_gallery = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # -> one 2-D embedding per identity
t1_query = np.array([0.15, 0.10])  # -> query embedding near Ava
t1_claimed = "Ava"  # -> claimed identity for verification
t1_threshold = 0.25  # -> accept if distance is at most 0.25
t1_claim_index = int(np.where(t1_names == t1_claimed)[0][0])  # -> 0
t1_diffs = t1_gallery - t1_query  # -> query-to-gallery vector differences
t1_squared_diffs = t1_diffs ** 2  # -> squared differences
t1_squared_distances = np.sum(t1_squared_diffs, axis=1)  # -> [0.0325, 0.7325, 0.8325, 1.5325]
t1_distances = np.sqrt(t1_squared_distances)  # -> [0.1803, 0.8559, 0.9124, 1.2379]
t1_verification_distance = t1_distances[t1_claim_index]  # -> 0.1803
t1_accept = t1_verification_distance <= t1_threshold  # -> True
t1_best_index = int(np.argmin(t1_distances))  # -> 0
t1_best_name = t1_names[t1_best_index]  # -> 'Ava'

print("rng seed:", 0)
print("names:", t1_names.tolist())
print("gallery embeddings:", t1_gallery.tolist())
print("query:", t1_query.tolist())
print("differences gallery - query:", np.round(t1_diffs, 3).tolist())
print("squared distances:", np.round(t1_squared_distances, 4).tolist())
print("L2 distances:", np.round(t1_distances, 3).tolist())
print("verification distance to claimed Ava:", round(float(t1_verification_distance), 3))
print("accept claim?:", bool(t1_accept))
print("recognition winner:", str(t1_best_name))

assert t1_accept is True or bool(t1_accept) is True
assert t1_best_name == "Ava"
assert np.allclose(np.round(t1_distances, 3), [0.18, 0.856, 0.912, 1.238])

t1_fig, t1_ax = plt.subplots(figsize=(6, 3.5))
t1_ax.bar(t1_names, t1_distances, color=["seagreen", "steelblue", "orange", "gray"])
t1_ax.axhline(t1_threshold, color="red", linestyle="--", label="verification threshold")
t1_ax.set_title("Verification threshold vs recognition nearest neighbor")
t1_ax.set_ylabel("embedding distance")
t1_ax.legend()
plt.show()

▶ What you'll see: Ava's distance is below the threshold and also the smallest gallery distance, so verification accepts and recognition returns Ava.

### ✍️ Toy 2 · Cosine similarity and unit normalization

Cosine similarity compares vector direction after accounting for vector length. Unit normalization makes that geometry visible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t2_rng = np.random.default_rng(0)  # -> reproducible generator seeded with 0
t2_a = np.array([3.0, 4.0])  # -> first embedding
t2_b = np.array([4.0, 0.0])  # -> second embedding
t2_dot = np.dot(t2_a, t2_b)  # -> 12.0
t2_norm_a = np.linalg.norm(t2_a)  # -> 5.0
t2_norm_b = np.linalg.norm(t2_b)  # -> 4.0
t2_norm_product = t2_norm_a * t2_norm_b  # -> 20.0
t2_cosine = t2_dot / t2_norm_product  # -> 0.6
t2_angle = np.degrees(np.arccos(t2_cosine))  # -> 53.13010235415599
t2_a_unit = t2_a / t2_norm_a  # -> [0.6, 0.8]
t2_b_unit = t2_b / t2_norm_b  # -> [1.0, 0.0]

print("rng seed:", 0)
print("a:", t2_a.tolist())
print("b:", t2_b.tolist())
print("dot product:", float(t2_dot))
print("norm a:", float(t2_norm_a))
print("norm b:", float(t2_norm_b))
print("norm product:", float(t2_norm_product))
print("cosine similarity:", round(float(t2_cosine), 3))
print("angle degrees:", round(float(t2_angle), 3))
print("unit a:", t2_a_unit.tolist())
print("unit b:", t2_b_unit.tolist())

assert np.isclose(t2_cosine, 0.6)
assert np.allclose(t2_a_unit, [0.6, 0.8])
assert np.allclose(t2_b_unit, [1.0, 0.0])

t2_fig, t2_ax = plt.subplots(figsize=(4.5, 4))
t2_ax.quiver([0, 0], [0, 0], [t2_a_unit[0], t2_b_unit[0]], [t2_a_unit[1], t2_b_unit[1]], angles="xy", scale_units="xy", scale=1, color=["tab:blue", "tab:orange"])
t2_ax.set_xlim(-0.1, 1.1)
t2_ax.set_ylim(-0.1, 1.1)
t2_ax.set_aspect("equal", adjustable="box")
t2_ax.grid(True, alpha=0.3)
t2_ax.set_title("Cosine similarity after unit normalization")
t2_ax.set_xlabel("embedding coordinate 1")
t2_ax.set_ylabel("embedding coordinate 2")
plt.show()

▶ What you'll see: the normalized arrows have cosine similarity 0.6, equivalent to an angle of about 53.13°.

### ✍️ Toy 3 · One-shot nearest-neighbor enrollment

A Siamese embedding model can enroll a new identity by storing one reference vector and doing nearest-neighbor lookup.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t3_rng = np.random.default_rng(0)  # -> reproducible generator seeded with 0
t3_names = np.array(["Ava", "Ben", "Dora (new)"])  # -> one-shot gallery includes a new identity
t3_gallery = np.array([[0.10, 0.20], [0.80, 0.20], [0.50, 0.70]])  # -> one reference embedding per person
t3_query = np.array([0.52, 0.68])  # -> query from Dora
t3_diffs = t3_gallery - t3_query  # -> vector differences to the query
t3_squared = t3_diffs ** 2  # -> squared coordinate errors
t3_squared_distances = np.sum(t3_squared, axis=1)  # -> [0.4072, 0.3092, 0.0008]
t3_distances = np.sqrt(t3_squared_distances)  # -> [0.638, 0.556, 0.028]
t3_best_index = int(np.argmin(t3_distances))  # -> 2
t3_prediction = t3_names[t3_best_index]  # -> 'Dora (new)'

print("rng seed:", 0)
print("gallery:", t3_gallery.tolist())
print("query:", t3_query.tolist())
print("differences:", np.round(t3_diffs, 3).tolist())
print("squared distances:", np.round(t3_squared_distances, 4).tolist())
print("L2 distances:", np.round(t3_distances, 3).tolist())
print("nearest index:", t3_best_index)
print("one-shot prediction:", str(t3_prediction))

assert t3_prediction == "Dora (new)"
assert np.allclose(np.round(t3_distances, 3), [0.638, 0.556, 0.028])

t3_fig, t3_ax = plt.subplots(figsize=(5, 4))
t3_ax.scatter(t3_gallery[:, 0], t3_gallery[:, 1], s=90, label="stored references")
t3_ax.scatter([t3_query[0]], [t3_query[1]], marker="*", s=180, color="black", label="query")
for t3_i, t3_name in enumerate(t3_names):
    t3_ax.text(t3_gallery[t3_i, 0] + 0.01, t3_gallery[t3_i, 1] + 0.01, str(t3_name))
t3_ax.plot([t3_query[0], t3_gallery[t3_best_index, 0]], [t3_query[1], t3_gallery[t3_best_index, 1]], "k--")
t3_ax.set_title("One stored embedding is enough for lookup")
t3_ax.set_xlabel("embedding coordinate 1")
t3_ax.set_ylabel("embedding coordinate 2")
t3_ax.legend()
plt.show()

▶ What you'll see: the query star is almost on top of Dora's one enrolled reference, so nearest-neighbor recognition returns the new identity.

### ✍️ Toy 4 · Triplet loss and a margin-fixing update

Triplet loss is positive when the negative is not at least the margin farther from the anchor than the positive is.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t4_rng = np.random.default_rng(0)  # -> reproducible generator seeded with 0
t4_anchor = np.array([0.0, 0.0])  # -> anchor embedding
t4_positive = np.array([1.0, 0.0])  # -> same-identity positive
t4_negative = np.array([1.2, 0.1])  # -> different-identity negative, still too close
t4_margin = 0.50  # -> required extra distance
t4_d_ap_before = np.linalg.norm(t4_anchor - t4_positive)  # -> 1.0
t4_d_an_before = np.linalg.norm(t4_anchor - t4_negative)  # -> 1.2041594578792296
t4_loss_before = max(t4_d_ap_before - t4_d_an_before + t4_margin, 0.0)  # -> 0.2958405421207704
t4_learning_rate = 0.40  # -> visible toy update size
t4_positive_after = t4_positive - t4_learning_rate * (t4_positive - t4_anchor)  # -> [0.6, 0.0]
t4_negative_after = t4_negative + t4_learning_rate * (t4_negative - t4_anchor)  # -> [1.68, 0.14]
t4_d_ap_after = np.linalg.norm(t4_anchor - t4_positive_after)  # -> 0.6
t4_d_an_after = np.linalg.norm(t4_anchor - t4_negative_after)  # -> 1.6858232410309213
t4_loss_after = max(t4_d_ap_after - t4_d_an_after + t4_margin, 0.0)  # -> 0.0

print("rng seed:", 0)
print("anchor:", t4_anchor.tolist())
print("positive before:", t4_positive.tolist())
print("negative before:", t4_negative.tolist())
print("d(anchor, positive) before:", round(float(t4_d_ap_before), 3))
print("d(anchor, negative) before:", round(float(t4_d_an_before), 3))
print("triplet loss before:", round(float(t4_loss_before), 3))
print("positive after:", t4_positive_after.tolist())
print("negative after:", np.round(t4_negative_after, 3).tolist())
print("d(anchor, positive) after:", round(float(t4_d_ap_after), 3))
print("d(anchor, negative) after:", round(float(t4_d_an_after), 3))
print("triplet loss after:", round(float(t4_loss_after), 3))

assert np.isclose(round(float(t4_loss_before), 6), 0.295841)
assert t4_loss_after == 0.0

t4_fig, t4_axes = plt.subplots(1, 2, figsize=(8, 3.5))
t4_axes[0].bar(["d(A,P)", "d(A,N)"], [t4_d_ap_before, t4_d_an_before], alpha=0.5, label="before")
t4_axes[0].bar(["d(A,P)", "d(A,N)"], [t4_d_ap_after, t4_d_an_after], alpha=0.8, label="after")
t4_axes[0].axhline(t4_d_ap_after + t4_margin, color="red", linestyle="--", label="after d(A,P)+margin")
t4_axes[0].set_title("Triplet margin")
t4_axes[0].legend(fontsize=8)
t4_axes[1].scatter([t4_anchor[0]], [t4_anchor[1]], color="black", s=120, label="anchor")
t4_axes[1].scatter([t4_positive[0], t4_negative[0]], [t4_positive[1], t4_negative[1]], color=["green", "red"], alpha=0.45, s=80, label="before")
t4_axes[1].scatter([t4_positive_after[0], t4_negative_after[0]], [t4_positive_after[1], t4_negative_after[1]], color=["green", "red"], marker="*", s=140, label="after")
t4_axes[1].set_title("Pull positive, push negative")
t4_axes[1].legend(fontsize=8)
t4_fig.tight_layout()
plt.show()

▶ What you'll see: before the update the margin is violated; after pulling the positive in and pushing the negative out, the triplet loss is zero.

### ✍️ Toy 5 · Triplet mining labels

Triplet mining categorizes negatives as hard, semi-hard, or easy by comparing their distance to the anchor-positive distance plus the margin.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t5_rng = np.random.default_rng(0)  # -> reproducible generator seeded with 0
t5_anchor = np.array([0.0, 0.0])  # -> anchor embedding
t5_positive = np.array([0.30, 0.0])  # -> positive embedding
t5_negatives = np.array([[0.20, 0.0], [0.50, 0.0], [1.00, 0.0]])  # -> hard, semi-hard, easy negatives
t5_negative_names = np.array(["hard", "semi-hard", "easy"])  # -> expected mining categories
t5_margin = 0.40  # -> triplet margin
t5_d_ap = np.linalg.norm(t5_anchor - t5_positive)  # -> 0.3
t5_d_an = np.linalg.norm(t5_negatives - t5_anchor, axis=1)  # -> [0.2, 0.5, 1.0]
t5_margin_boundary = t5_d_ap + t5_margin  # -> 0.7
t5_losses = np.maximum(t5_d_ap - t5_d_an + t5_margin, 0.0)  # -> [0.5, 0.2, 0.0]
t5_is_hard = t5_d_an <= t5_d_ap  # -> [True, False, False]
t5_is_semi_hard = (t5_d_an > t5_d_ap) & (t5_d_an < t5_margin_boundary)  # -> [False, True, False]
t5_is_easy = t5_d_an >= t5_margin_boundary  # -> [False, False, True]
t5_labels = np.where(t5_is_hard, "hard", np.where(t5_is_semi_hard, "semi-hard", "easy"))  # -> ['hard', 'semi-hard', 'easy']

print("rng seed:", 0)
print("anchor:", t5_anchor.tolist())
print("positive:", t5_positive.tolist())
print("negatives:", t5_negatives.tolist())
print("d(A,P):", round(float(t5_d_ap), 3))
print("d(A,N):", t5_d_an.tolist())
print("d(A,P)+margin:", round(float(t5_margin_boundary), 3))
print("triplet losses:", t5_losses.tolist())
print("mining labels:", t5_labels.tolist())

assert np.allclose(t5_losses, [0.5, 0.2, 0.0])
assert t5_labels.tolist() == ["hard", "semi-hard", "easy"]

t5_fig, t5_ax = plt.subplots(figsize=(6, 3.5))
t5_ax.bar(t5_negative_names, t5_d_an, color=["tomato", "gold", "seagreen"])
t5_ax.axhline(t5_d_ap, color="black", linestyle="--", label="d(A,P)")
t5_ax.axhline(t5_margin_boundary, color="red", linestyle="--", label="d(A,P)+margin")
t5_ax.set_title("Triplet mining categories")
t5_ax.set_ylabel("negative distance")
t5_ax.legend()
plt.show()

▶ What you'll see: the hard negative is closer than the positive, the semi-hard one violates the margin, and the easy one has zero loss.

### ✍️ Toy 6 · Gram matrix and style loss

The Gram matrix sums channel co-activations over spatial positions. Style loss compares Gram matrices, not pixel-aligned features.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t6_rng = np.random.default_rng(0)  # -> reproducible generator seeded with 0
t6_style = np.array([[[1.0, 0.0], [0.0, 1.0]], [[1.0, 1.0], [0.0, 1.0]]])  # -> 2x2x2 style activations
t6_generated = np.array([[[1.0, 1.0], [0.0, 1.0]], [[1.0, 0.0], [1.0, 1.0]]])  # -> 2x2x2 generated activations
t6_h, t6_w, t6_c = t6_generated.shape  # -> (2, 2, 2)
t6_style_features = t6_style.reshape(t6_h * t6_w, t6_c)  # -> four spatial rows by two channels
t6_generated_features = t6_generated.reshape(t6_h * t6_w, t6_c)  # -> four spatial rows by two channels
t6_style_gram = t6_style_features.T @ t6_style_features  # -> [[2.0, 1.0], [1.0, 3.0]]
t6_generated_gram = t6_generated_features.T @ t6_generated_features  # -> [[3.0, 2.0], [2.0, 3.0]]
t6_gram_diff = t6_style_gram - t6_generated_gram  # -> [[-1.0, -1.0], [-1.0, 0.0]]
t6_squared_diff = t6_gram_diff ** 2  # -> [[1.0, 1.0], [1.0, 0.0]]
t6_denominator = float((2 * t6_h * t6_w * t6_c) ** 2)  # -> 256.0
t6_style_loss = np.sum(t6_squared_diff) / t6_denominator  # -> 0.01171875

print("rng seed:", 0)
print("style features:", t6_style_features.tolist())
print("generated features:", t6_generated_features.tolist())
print("style Gram:", t6_style_gram.tolist())
print("generated Gram:", t6_generated_gram.tolist())
print("Gram difference:", t6_gram_diff.tolist())
print("squared difference:", t6_squared_diff.tolist())
print("denominator:", t6_denominator)
print("style loss:", float(t6_style_loss))

assert np.array_equal(t6_style_gram, [[2.0, 1.0], [1.0, 3.0]])
assert np.array_equal(t6_generated_gram, [[3.0, 2.0], [2.0, 3.0]])
assert np.isclose(t6_style_loss, 3.0 / 256.0)

t6_fig, t6_axes = plt.subplots(1, 2, figsize=(6, 3))
t6_image0 = t6_axes[0].imshow(t6_style_gram, cmap="magma", vmin=0, vmax=3)
t6_axes[0].set_title("style Gram")
t6_image1 = t6_axes[1].imshow(t6_generated_gram, cmap="magma", vmin=0, vmax=3)
t6_axes[1].set_title("generated Gram")
t6_fig.colorbar(t6_image1, ax=t6_axes.ravel().tolist(), fraction=0.046)
plt.show()

▶ What you'll see: two 2×2 Gram heatmaps and a style loss of 3 / 256 = 0.01171875.

### ✍️ Toy 7 · Content loss and weighted style-transfer objective

Content loss compares activations at the same positions, while the total objective weights content loss against style loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t7_rng = np.random.default_rng(0)  # -> reproducible generator seeded with 0
t7_content = np.array([[[1.0, 0.0], [0.0, 1.0]], [[1.0, 1.0], [0.0, 0.0]]])  # -> 2x2x2 content activations
t7_style = np.array([[[1.0, 0.0], [0.0, 1.0]], [[1.0, 1.0], [0.0, 1.0]]])  # -> 2x2x2 style activations
t7_generated = np.array([[[0.8, 0.1], [0.2, 0.9]], [[0.9, 0.8], [0.1, 0.2]]])  # -> 2x2x2 generated activations
t7_diff = t7_generated - t7_content  # -> generated minus content
t7_squared_diff = t7_diff ** 2  # -> squared spatial activation errors
t7_content_loss = 0.5 * np.sum(t7_squared_diff)  # -> 0.1
t7_h, t7_w, t7_c = t7_generated.shape  # -> (2, 2, 2)
t7_style_features = t7_style.reshape(t7_h * t7_w, t7_c)  # -> style features
t7_generated_features = t7_generated.reshape(t7_h * t7_w, t7_c)  # -> generated features
t7_style_gram = t7_style_features.T @ t7_style_features  # -> [[2.0, 1.0], [1.0, 3.0]]
t7_generated_gram = t7_generated_features.T @ t7_generated_features  # -> [[1.5, 1.0], [1.0, 1.5]]
t7_style_loss = np.sum((t7_style_gram - t7_generated_gram) ** 2) / float((2 * t7_h * t7_w * t7_c) ** 2)  # -> 0.009765625
t7_alpha = 1.0  # -> content weight
t7_beta = 20.0  # -> style weight
t7_weighted_content = t7_alpha * t7_content_loss  # -> 0.1
t7_weighted_style = t7_beta * t7_style_loss  # -> 0.1953125
t7_total_loss = t7_weighted_content + t7_weighted_style  # -> 0.2953125

print("rng seed:", 0)
print("generated - content:", np.round(t7_diff, 3).tolist())
print("squared difference:", np.round(t7_squared_diff, 3).tolist())
print("content loss:", round(float(t7_content_loss), 4))
print("style Gram:", np.round(t7_style_gram, 3).tolist())
print("generated Gram:", np.round(t7_generated_gram, 3).tolist())
print("style loss:", round(float(t7_style_loss), 6))
print("weighted content:", round(float(t7_weighted_content), 4))
print("weighted style:", round(float(t7_weighted_style), 4))
print("total loss:", round(float(t7_total_loss), 4))

assert np.isclose(t7_content_loss, 0.1)
assert np.isclose(t7_style_loss, 0.009765625)
assert np.isclose(t7_total_loss, 0.2953125)

t7_fig, t7_axes = plt.subplots(1, 3, figsize=(8, 3))
t7_axes[0].imshow(np.clip(t7_content[..., 0], 0.0, 1.0), cmap="Blues", vmin=0.0, vmax=1.0)
t7_axes[0].set_title("content")
t7_axes[0].axis("off")
t7_axes[1].imshow(np.clip(t7_generated[..., 0], 0.0, 1.0), cmap="Blues", vmin=0.0, vmax=1.0)
t7_axes[1].set_title("generated")
t7_axes[1].axis("off")
t7_axes[2].bar(["content", "style×β"], [t7_weighted_content, t7_weighted_style], color=["steelblue", "salmon"])
t7_axes[2].set_title("objective pieces")
t7_fig.tight_layout()
plt.show()

▶ What you'll see: a content-vs-generated image comparison and a bar chart where weighted style contributes about 0.1953 to the total.

### ✍️ Toy 8 · One gradient step for style transfer

A style-transfer update subtracts a gradient made from a content term plus a Gram-matrix style term, then re-evaluates the loss.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t8_rng = np.random.default_rng(0)  # -> reproducible generator seeded with 0
t8_content = np.array([[[1.0, 0.0], [0.0, 1.0]], [[1.0, 1.0], [0.0, 0.0]]])  # -> content activations
t8_style = np.array([[[1.0, 0.0], [0.0, 1.0]], [[1.0, 1.0], [0.0, 1.0]]])  # -> style activations
t8_generated = np.array([[[0.8, 0.1], [0.2, 0.9]], [[0.9, 0.8], [0.1, 0.2]]])  # -> starting generated activations
t8_alpha = 1.0  # -> content weight
t8_beta = 20.0  # -> style weight
t8_learning_rate = 0.20  # -> one safe gradient step
t8_h, t8_w, t8_c = t8_generated.shape  # -> (2, 2, 2)
t8_content_grad = t8_generated - t8_content  # -> content gradient
t8_generated_features = t8_generated.reshape(t8_h * t8_w, t8_c)  # -> generated features
t8_style_features = t8_style.reshape(t8_h * t8_w, t8_c)  # -> style features
t8_generated_gram = t8_generated_features.T @ t8_generated_features  # -> [[1.5, 1.0], [1.0, 1.5]]
t8_style_gram = t8_style_features.T @ t8_style_features  # -> [[2.0, 1.0], [1.0, 3.0]]
t8_denominator = float((2 * t8_h * t8_w * t8_c) ** 2)  # -> 256.0
t8_style_grad_flat = (4.0 / t8_denominator) * t8_generated_features @ (t8_generated_gram - t8_style_gram)  # -> Gram-style gradient, flattened
t8_style_grad = t8_style_grad_flat.reshape(t8_h, t8_w, t8_c)  # -> Gram-style gradient tensor
t8_total_grad = t8_alpha * t8_content_grad + t8_beta * t8_style_grad  # -> weighted total gradient
t8_next_generated = np.clip(t8_generated - t8_learning_rate * t8_total_grad, 0.0, 1.0)  # -> updated activations
t8_old_content_loss = 0.5 * np.sum((t8_generated - t8_content) ** 2)  # -> 0.1
t8_old_style_loss = np.sum((t8_style_gram - t8_generated_gram) ** 2) / t8_denominator  # -> 0.009765625
t8_old_total = t8_alpha * t8_old_content_loss + t8_beta * t8_old_style_loss  # -> 0.2953125
t8_next_features = t8_next_generated.reshape(t8_h * t8_w, t8_c)  # -> updated generated features
t8_next_gram = t8_next_features.T @ t8_next_features  # -> updated Gram matrix
t8_new_content_loss = 0.5 * np.sum((t8_next_generated - t8_content) ** 2)  # -> about 0.0712
t8_new_style_loss = np.sum((t8_style_gram - t8_next_gram) ** 2) / t8_denominator  # -> about 0.00445
t8_new_total = t8_alpha * t8_new_content_loss + t8_beta * t8_new_style_loss  # -> about 0.1602

print("rng seed:", 0)
print("content gradient:", np.round(t8_content_grad, 4).tolist())
print("style Gram:", np.round(t8_style_gram, 4).tolist())
print("generated Gram:", np.round(t8_generated_gram, 4).tolist())
print("style gradient:", np.round(t8_style_grad, 4).tolist())
print("total gradient:", np.round(t8_total_grad, 4).tolist())
print("updated generated:", np.round(t8_next_generated, 4).tolist())
print("old total loss:", round(float(t8_old_total), 4))
print("new total loss:", round(float(t8_new_total), 4))

assert np.isclose(t8_old_total, 0.2953125)
assert np.isclose(round(float(t8_new_total), 4), 0.1602)
assert t8_new_total < t8_old_total

t8_fig, t8_axes = plt.subplots(1, 3, figsize=(8, 3))
t8_axes[0].imshow(np.clip(t8_generated[..., 0], 0.0, 1.0), cmap="Blues", vmin=0.0, vmax=1.0)
t8_axes[0].set_title("before")
t8_axes[0].axis("off")
t8_axes[1].imshow(np.clip(t8_next_generated[..., 0], 0.0, 1.0), cmap="Blues", vmin=0.0, vmax=1.0)
t8_axes[1].set_title("after one step")
t8_axes[1].axis("off")
t8_axes[2].bar(["old", "new"], [t8_old_total, t8_new_total], color=["gray", "seagreen"])
t8_axes[2].set_title("total loss")
t8_fig.tight_layout()
plt.show()

▶ What you'll see: one gradient step lowers the weighted style-transfer objective from about 0.2953 to about 0.1602.

## 0. Step-by-Step Worked Example — Start Here (Beginner Friendly)

> 🧑‍🎓 **New to this topic? Start here.** This is a gentle, fully runnable walkthrough that
> builds up *every* idea in this lesson one tiny step at a time. Each step **prints** the
> numbers it computes and **draws a picture** so you can *see* what is happening. Run the
> cells in order from top to bottom. Nothing here needs the internet or any downloaded data.

### The Big Picture — What You'll Learn

In plain terms, here is what the steps below will show you:

- **Verification** makes one thresholded 1-to-1 decision, while **recognition** searches a whole gallery.
- **Siamese embeddings** make **one-shot** enrollment possible by comparing a query to one stored vector.
- **Triplet loss** reduces the anchor-positive distance and increases the anchor-negative distance past a margin.
- **Neural style transfer** combines content loss with Gram-matrix style loss in one weighted objective.

Everything below (starting at **§1 Overview**) develops these same ideas with fuller examples,
threshold sweeps, triplet mining, and toy style-transfer optimization.

**What we will build, step by step:**
1. **Face verification versus face recognition** — one claimed identity vs. a whole gallery.
2. **Siamese embeddings and one-shot learning** — add a new person with one example.
3. **Triplet loss** — pull positives close and push negatives past a margin.
4. **Neural style transfer** — content loss, Gram-matrix style loss, and the weighted objective.

### Step 0 — Set up our tools

We import NumPy (vectors + tiny synthetic arrays) and Matplotlib (pictures). We fix a random
**seed** so every run gives the same printed numbers, then define a tiny `log()` helper so each
line of output has a clear label.

In [ ]:
import numpy as np                       # NumPy: tiny embeddings, distances, feature maps, and matrix math.
import matplotlib.pyplot as plt          # Matplotlib: draw distance bars, embedding maps, and Gram heatmaps.

np.random.seed(0)                         # Fix the seed so every run prints the SAME numbers.
plt.rcParams["figure.figsize"] = (7, 4)   # A comfortable default plot size.

def log(label, value):                    # A tiny logger so each printed line explains itself.
    print(f"[{label}] {value}")           # Format is: [what this is] the value.

log("setup", "tools ready — NumPy + Matplotlib imported, seed fixed to 0")

▶ What you'll see: one line confirming the tools are ready.

### Step 1 — Face verification versus face recognition

Verification asks a **1-to-1** question: "Is this query really the claimed person?" Recognition
asks a **1-to-many** question: "Which enrolled person is closest?" Both use embedding distances,
but the decision rule is different.

In [ ]:
gallery_names_demo = np.array(["Ava", "Ben", "Cy"])                         # Three enrolled identities in the gallery.
gallery_embeddings_demo = np.array([[0.10, 0.20], [0.82, 0.18], [0.22, 0.82]]) # One 2-D face embedding per identity.
query_embedding_demo = np.array([0.16, 0.24])                                # A query face embedding that should look like Ava.
claimed_name_demo = "Ava"                                                    # The identity being claimed in verification.
threshold_demo = 0.12                                                        # Accept the claim only below this distance.

claimed_index_demo = int(np.where(gallery_names_demo == claimed_name_demo)[0][0]) # Find Ava's row in the gallery.
verification_distance_demo = np.linalg.norm(query_embedding_demo - gallery_embeddings_demo[claimed_index_demo]) # One 1-to-1 distance.
verification_accept_demo = verification_distance_demo <= threshold_demo       # Turn that one distance into accept/reject.

recognition_distances_demo = np.linalg.norm(gallery_embeddings_demo - query_embedding_demo, axis=1) # One distance to every gallery entry.
best_index_demo = int(np.argmin(recognition_distances_demo))                 # Recognition chooses the nearest gallery row.
best_name_demo = gallery_names_demo[best_index_demo]                         # Convert the nearest row into an identity name.

for name_demo, distance_demo in zip(gallery_names_demo, recognition_distances_demo): # Print every gallery comparison.
    log(f"distance to {name_demo}", round(float(distance_demo), 3))           # Show one recognition distance.
log("verification claim", f"{claimed_name_demo} -> {'ACCEPT' if verification_accept_demo else 'REJECT'}") # Show the 1-to-1 decision.
log("recognition winner", best_name_demo)                                    # Show the 1-to-many nearest identity.

plt.bar(gallery_names_demo, recognition_distances_demo, color=["seagreen", "steelblue", "orange"]) # Draw all query-to-gallery distances.
plt.axhline(threshold_demo, color="red", linestyle="--", label="verification threshold") # Draw the accept/reject cutoff.
plt.scatter([best_name_demo], [recognition_distances_demo[best_index_demo]], color="black", zorder=3, label="nearest") # Mark the winner.
plt.ylabel("Euclidean embedding distance")                                  # Label the distance scale.
plt.title("Verification uses one threshold; recognition searches the gallery") # Title the comparison.
plt.legend()                                                                # Explain the threshold and nearest marker.
plt.show()                                                                  # Render the plot.

▶ What you'll see: Ava is accepted in verification and also wins the recognition search because her distance bar is shortest.

### Step 2 — Siamese embeddings and one-shot learning

A Siamese model uses the **same encoder** for both images, so all outputs live in one shared
embedding space. That makes one-shot learning possible: to enroll a new person, store one
reference embedding and compare future queries to it.

In [ ]:
one_shot_names_demo = np.array(["Ava", "Ben", "Dora (new)"])                 # Dora is a newly enrolled identity.
one_shot_gallery_demo = np.array([[0.10, 0.20], [0.82, 0.18], [0.48, 0.76]])  # Store just one reference embedding per person.
one_shot_query_demo = np.array([0.52, 0.72])                                 # A query embedding from Dora.
one_shot_distances_demo = np.linalg.norm(one_shot_gallery_demo - one_shot_query_demo, axis=1) # Compare the query to all references.
one_shot_index_demo = int(np.argmin(one_shot_distances_demo))                # Pick the nearest reference.
one_shot_prediction_demo = one_shot_names_demo[one_shot_index_demo]          # Translate nearest row into a predicted name.

for name_demo, distance_demo in zip(one_shot_names_demo, one_shot_distances_demo): # Print each one-shot comparison.
    log(f"one-shot distance to {name_demo}", round(float(distance_demo), 3))  # Show one distance.
log("one-shot prediction", one_shot_prediction_demo)                         # Show the nearest-neighbor prediction.

plt.scatter(one_shot_gallery_demo[:, 0], one_shot_gallery_demo[:, 1], s=90, label="gallery references") # Draw enrolled embeddings.
plt.scatter(one_shot_query_demo[0], one_shot_query_demo[1], s=160, marker="*", color="black", label="query") # Draw the query as a star.
for index_demo, name_demo in enumerate(one_shot_names_demo):                 # Label each reference point.
    plt.text(one_shot_gallery_demo[index_demo, 0] + 0.01, one_shot_gallery_demo[index_demo, 1] + 0.01, name_demo, fontsize=9) # Add a name label.
plt.plot([one_shot_query_demo[0], one_shot_gallery_demo[one_shot_index_demo, 0]], [one_shot_query_demo[1], one_shot_gallery_demo[one_shot_index_demo, 1]], "k--") # Connect query to nearest reference.
plt.xlabel("embedding coordinate 1")                                        # Label the horizontal coordinate.
plt.ylabel("embedding coordinate 2")                                        # Label the vertical coordinate.
plt.title("One stored embedding is enough for one-shot lookup")             # Title the geometry plot.
plt.legend()                                                                # Explain point markers.
plt.show()                                                                  # Render the plot.

▶ What you'll see: the query star lands closest to Dora's single stored example, so the new identity is recognized without retraining.

### Step 3 — Triplet loss

Triplet loss looks at an **anchor** image, a same-person **positive**, and a different-person
**negative**. The loss is positive when the negative is not at least margin $\alpha$ farther
than the positive, so training pulls positives inward and pushes negatives outward.

In [ ]:
anchor_demo = np.array([0.15, 0.20])                                        # Anchor embedding for one identity.
positive_demo = np.array([0.52, 0.35])                                      # Positive embedding from the same identity.
negative_demo = np.array([0.62, 0.26])                                      # Negative embedding from a different identity.
margin_demo = 0.30                                                          # Required extra separation for the negative.

d_ap_before_demo = np.linalg.norm(anchor_demo - positive_demo)              # Distance from anchor to positive before training.
d_an_before_demo = np.linalg.norm(anchor_demo - negative_demo)              # Distance from anchor to negative before training.
loss_before_demo = max(d_ap_before_demo - d_an_before_demo + margin_demo, 0.0) # Hinge triplet loss before training.

learning_rate_demo = 0.35                                                   # A visible toy update size.
positive_after_demo = positive_demo - learning_rate_demo * (positive_demo - anchor_demo) # Pull positive toward the anchor.
negative_after_demo = negative_demo + learning_rate_demo * (negative_demo - anchor_demo) # Push negative away from the anchor.
d_ap_after_demo = np.linalg.norm(anchor_demo - positive_after_demo)         # Distance from anchor to positive after the update.
d_an_after_demo = np.linalg.norm(anchor_demo - negative_after_demo)         # Distance from anchor to negative after the update.
loss_after_demo = max(d_ap_after_demo - d_an_after_demo + margin_demo, 0.0) # Hinge triplet loss after the update.

log("before d(anchor, positive)", round(float(d_ap_before_demo), 3))         # Print same-identity distance before.
log("before d(anchor, negative)", round(float(d_an_before_demo), 3))         # Print different-identity distance before.
log("before triplet loss", round(float(loss_before_demo), 3))               # Print margin violation before.
log("after d(anchor, positive)", round(float(d_ap_after_demo), 3))           # Print same-identity distance after.
log("after d(anchor, negative)", round(float(d_an_after_demo), 3))           # Print different-identity distance after.
log("after triplet loss", round(float(loss_after_demo), 3))                 # Print margin violation after.

fig_demo, axes_demo = plt.subplots(1, 2, figsize=(10, 4))                   # Create distance and geometry panels.
axes_demo[0].bar(["d(A,P)", "d(A,N)"], [d_ap_before_demo, d_an_before_demo], alpha=0.45, label="before") # Draw before distances.
axes_demo[0].bar(["d(A,P)", "d(A,N)"], [d_ap_after_demo, d_an_after_demo], alpha=0.75, label="after") # Draw after distances.
axes_demo[0].axhline(d_ap_after_demo + margin_demo, color="red", linestyle="--", label="after positive + margin") # Show target separation.
axes_demo[0].set_ylabel("embedding distance")                              # Label the distance axis.
axes_demo[0].set_title("Triplet margin before and after")                  # Title the bar panel.
axes_demo[0].legend()                                                       # Explain before/after bars.
axes_demo[1].scatter(anchor_demo[0], anchor_demo[1], s=140, color="black", label="anchor") # Draw the anchor.
axes_demo[1].scatter(positive_demo[0], positive_demo[1], s=90, color="green", alpha=0.4, label="positive before") # Draw original positive.
axes_demo[1].scatter(negative_demo[0], negative_demo[1], s=90, color="red", alpha=0.4, label="negative before") # Draw original negative.
axes_demo[1].scatter(positive_after_demo[0], positive_after_demo[1], s=140, marker="*", color="green", label="positive after") # Draw moved positive.
axes_demo[1].scatter(negative_after_demo[0], negative_after_demo[1], s=140, marker="*", color="red", label="negative after") # Draw moved negative.
axes_demo[1].plot([positive_demo[0], positive_after_demo[0]], [positive_demo[1], positive_after_demo[1]], "g--") # Show positive movement.
axes_demo[1].plot([negative_demo[0], negative_after_demo[0]], [negative_demo[1], negative_after_demo[1]], "r--") # Show negative movement.
axes_demo[1].set_title("Positive moves in; negative moves out")             # Title the embedding panel.
axes_demo[1].legend(fontsize=8)                                             # Explain point markers.
plt.tight_layout()                                                          # Prevent panel labels from overlapping.
plt.show()                                                                  # Render both panels.

▶ What you'll see: after the toy update, the positive is closer, the negative is farther, and the triplet loss drops to zero.

### Step 4 — Neural style transfer: content loss, style loss, and Gram matrix

Neural style transfer optimizes a generated image $G$ by balancing two wishes: keep feature
maps close to the content image $C$, but match the style image $S$ through a **Gram matrix** of
channel co-activations. The weights $\alpha$ and $\beta$ decide the content/style tradeoff.

In [ ]:
grid_y_demo, grid_x_demo = np.mgrid[0:4, 0:4]                               # Build a tiny 4-by-4 coordinate grid.
content_base_demo = ((grid_x_demo == 1) | (grid_y_demo == 2)).astype(float)  # Make a simple cross-like content shape.
style_base_demo = ((grid_x_demo + grid_y_demo) % 2).astype(float)            # Make a checkerboard style texture.
content_demo = np.stack([content_base_demo, np.roll(content_base_demo, 1, axis=0), np.roll(content_base_demo, 1, axis=1)], axis=2) # Create 3 content channels.
style_demo = np.stack([style_base_demo, 1.0 - style_base_demo, np.roll(style_base_demo, 1, axis=0)], axis=2) # Create 3 style channels.
generated_demo = np.clip(0.65 * content_demo + 0.35 * style_demo, 0.0, 1.0)  # Start generated features between content and style.

def gram_matrix_demo(activation_demo):                                      # Define the Gram helper for H-by-W-by-C activations.
    h_demo, w_demo, c_demo = activation_demo.shape                          # Read height, width, and channels.
    features_demo = activation_demo.reshape(h_demo * w_demo, c_demo)        # Flatten spatial positions into rows.
    return features_demo.T @ features_demo                                  # Compute channel-by-channel co-activation totals.

content_loss_demo = 0.5 * np.sum((generated_demo - content_demo) ** 2)      # Content loss compares activations at the same positions.
gram_style_demo = gram_matrix_demo(style_demo)                              # Target style correlations.
gram_generated_demo = gram_matrix_demo(generated_demo)                      # Generated style correlations.
n_h_demo, n_w_demo, n_c_demo = generated_demo.shape                         # Read dimensions for the style-loss normalization.
style_loss_demo = np.sum((gram_style_demo - gram_generated_demo) ** 2) / float((2 * n_h_demo * n_w_demo * n_c_demo) ** 2) # Normalized Gram loss.
alpha_demo = 1.0                                                            # Content weight.
beta_demo = 30.0                                                            # Style weight.
total_loss_demo = alpha_demo * content_loss_demo + beta_demo * style_loss_demo # Weighted neural-style-transfer objective.

log("content loss", round(float(content_loss_demo), 4))                     # Print the content penalty.
log("style loss", round(float(style_loss_demo), 6))                         # Print the Gram-matrix style penalty.
log("weighted total loss", round(float(total_loss_demo), 4))                # Print the combined objective.
log("style Gram matrix", np.round(gram_style_demo, 2))                      # Print the style channel correlations.
log("generated Gram matrix", np.round(gram_generated_demo, 2))              # Print the generated channel correlations.

fig_demo, axes_demo = plt.subplots(2, 3, figsize=(10, 6))                   # Create image and matrix panels.
for ax_demo, tensor_demo, title_demo in zip(axes_demo[0], [content_demo, style_demo, generated_demo], ["content C", "style S", "generated G"]): # Loop over toy images.
    ax_demo.imshow(tensor_demo)                                             # Show each 3-channel tensor as an image.
    ax_demo.set_title(title_demo)                                           # Label the panel.
    ax_demo.axis("off")                                                     # Hide image axes.
for ax_demo, matrix_demo, title_demo in zip(axes_demo[1, :2], [gram_style_demo, gram_generated_demo], ["style Gram", "generated Gram"]): # Loop over Gram matrices.
    image_demo = ax_demo.imshow(matrix_demo, cmap="magma")                  # Draw one Gram heatmap.
    ax_demo.set_title(title_demo)                                           # Label the heatmap.
    ax_demo.set_xlabel("channel")                                           # Label Gram columns.
    ax_demo.set_ylabel("channel")                                           # Label Gram rows.
    plt.colorbar(image_demo, ax=ax_demo, fraction=0.046)                    # Add a small color scale.
axes_demo[1, 2].bar(["content", "style×β"], [alpha_demo * content_loss_demo, beta_demo * style_loss_demo], color=["steelblue", "salmon"]) # Compare weighted terms.
axes_demo[1, 2].set_title("weighted objective pieces")                      # Title the loss bar chart.
axes_demo[1, 2].set_ylabel("loss contribution")                             # Label contribution size.
plt.tight_layout()                                                          # Keep the panels readable.
plt.show()                                                                  # Render the style-transfer visualization.

▶ What you'll see: content is measured by spatial differences, style by Gram heatmaps, and the weighted bars show how $\alpha$ and $\beta$ form the final objective.

---

## 1. Overview

Face-recognition systems and neural style-transfer systems both reuse learned visual representations instead of treating raw pixels as the final object of interest. Face systems map images into embedding vectors and compare distances; style-transfer systems optimize a generated image so its internal activations preserve content while its channel correlations match style.

**One-line intuition:** one vision network can support two very different tasks: identity matching by embedding distance, and image generation by content/style losses.

## 2. Key Idea

### Face verification versus face recognition

**Face verification** asks a one-to-one question: "Is this query image the claimed person?" A system compares a query image to one enrolled reference image and accepts if the distance is below a threshold.

**Face recognition** asks a one-to-many question: "Which person in the database is this query?" A system compares the query to a gallery of $K$ identities, then returns the nearest identity or rejects the query if every distance is too large.

### Siamese embeddings and one-shot learning

A Siamese network uses the same encoder $f(\cdot)$ for both images in a pair. Instead of learning a separate classifier for every possible person, it learns an embedding space where same-identity images are close and different-identity images are far apart:

$$
d(\text{image 1},\text{image 2})=\left\|f(\text{image 1})-f(\text{image 2})\right\|_2.
$$

This is useful for one-shot learning because a new identity can be enrolled with only one or a few reference images; the learned similarity function does most of the work.

### Triplet loss

Triplet loss trains on three images: anchor $A$, positive $P$ from the same identity, and negative $N$ from a different identity. With margin $\alpha>0$,

$$
\ell(A,P,N)=\max\left(d(A,P)-d(A,N)+\alpha,0\right).
$$

The loss is zero only when the negative is at least $\alpha$ farther from the anchor than the positive is. Hard and semi-hard negatives matter because easy negatives already satisfy the margin and produce no gradient signal.

### Neural style transfer: content loss, style loss, Gram matrix

Neural style transfer starts with a content image $C$, a style image $S$, and a generated image $G$. A chosen layer's activation is denoted $a^{[l]}\in\mathbb{R}^{n_H\times n_W\times n_C}$.

The content loss keeps generated activations close to content activations:

$$
J_{\text{content}}(C,G)=\frac{1}{2}\left\|a^{[l](C)}-a^{[l](G)}\right\|^2.
$$

The style representation is the Gram matrix of channel correlations:

$$
G_{kk'}^{[l]}=\sum_{i=1}^{n_H^{[l]}}\sum_{j=1}^{n_W^{[l]}}a_{ijk}^{[l]}a_{ijk'}^{[l]}.
$$

A common single-layer style loss is

$$
J_{\text{style}}^{[l]}(S,G)=\frac{1}{(2n_Hn_Wn_C)^2}\left\|G^{[l](S)}-G^{[l](G)}\right\|_F^2.
$$

The generated image is optimized by the weighted objective

$$
J(G)=\alpha J_{\text{content}}(C,G)+\beta J_{\text{style}}(S,G).
$$

Higher $\alpha$ preserves more content structure; higher $\beta$ emphasizes style statistics.

## 3. Hands-on Notebook

### Setup

Run this first. Everything in this notebook is synthetic and CPU-friendly: no internet, no pretrained model downloads, and no GPU requirement.

In [ ]:
import numpy as np  # use NumPy for embeddings, distances, Gram matrices, and manual gradient steps.
import matplotlib.pyplot as plt  # use Matplotlib for all distance, embedding, and toy-image visualizations.
from math import acos  # use arccosine in the cosine-similarity warm-up angle calculation.
from math import degrees  # convert radians to degrees for an intuitive angle printout.
try:  # try to import ipywidgets for the final Colab slider experiment.
    from ipywidgets import interact  # create live controls when the notebook runs in Colab.
    from ipywidgets import FloatSlider  # create a distance-threshold slider for verification.
except ModuleNotFoundError:  # keep the notebook runnable in plain Python without widgets installed.
    class FloatSlider:  # define a minimal slider replacement that stores the default value.
        def __init__(self, value=0.0, min=0.0, max=1.0, step=0.1, description=""):  # accept the same keyword arguments used below.
            self.value = value  # store the value that the fallback interaction will pass to the function.
    def interact(function, **controls):  # define a fallback interaction function that runs once.
        values = {name: control.value for name, control in controls.items()}  # collect default control values into a dictionary.
        return function(**values)  # call the target function once so the notebook remains runnable.
np.random.seed(230)  # seed the legacy NumPy generator for reproducible examples.
RNG = np.random.default_rng(230)  # create a modern generator for all synthetic data below.
plt.style.use("seaborn-v0_8-whitegrid")  # choose a readable grid style for university-style plots.
COLORS = plt.cm.tab10.colors  # store a stable color cycle for identities and labels.

def l2_distance(a, b):  # define Euclidean distance between two embedding vectors.
    return float(np.linalg.norm(np.asarray(a) - np.asarray(b)))  # convert inputs to arrays and return a plain Python float.

def cosine_similarity(a, b):  # define cosine similarity between two embedding vectors.
    a = np.asarray(a)  # convert the first input to a NumPy array for vector operations.
    b = np.asarray(b)  # convert the second input to a NumPy array for vector operations.
    numerator = float(np.dot(a, b))  # compute the dot product that measures aligned direction.
    denominator = float(np.linalg.norm(a) * np.linalg.norm(b))  # compute the product of vector lengths for normalization.
    return numerator / denominator  # return the scale-free similarity in the range [-1, 1] for nonzero vectors.

def gram_matrix(activation):  # define the Gram matrix for an activation tensor shaped height by width by channels.
    h, w, c = activation.shape  # unpack the spatial and channel dimensions for clarity.
    features = activation.reshape(h * w, c)  # flatten spatial positions so each row is one location and each column is one channel.
    return features.T @ features  # multiply channels by channels to get channel-correlation totals.

def content_loss(content, generated):  # compute the CS230 content loss on toy activations.
    return 0.5 * float(np.sum((content - generated) ** 2))  # return one half times squared activation difference.

def style_loss(style, generated):  # compute a single-layer Gram-matrix style loss on toy activations.
    h, w, c = generated.shape  # read activation dimensions for the normalization constant.
    gram_style = gram_matrix(style)  # compute the target style channel correlations.
    gram_generated = gram_matrix(generated)  # compute the generated channel correlations.
    denominator = float((2 * h * w * c) ** 2)  # compute the standard style-loss normalization denominator.
    return float(np.sum((gram_style - gram_generated) ** 2) / denominator)  # return the normalized squared Gram difference.

def total_style_transfer_loss(content, style, generated, alpha=1.0, beta=10.0):  # combine content and style costs.
    c_loss = content_loss(content, generated)  # compute the content term before weighting.
    s_loss = style_loss(style, generated)  # compute the style term before weighting.
    total = alpha * c_loss + beta * s_loss  # combine losses using the requested tradeoff weights.
    return total, c_loss, s_loss  # return all pieces so plots can separate the terms.

def style_transfer_grad(content, style, generated, alpha=1.0, beta=10.0):  # compute a manual gradient for the toy generated activation.
    h, w, c = generated.shape  # unpack activation dimensions for reshaping and normalization.
    content_grad = generated - content  # differentiate one-half squared content loss with respect to generated activations.
    generated_features = generated.reshape(h * w, c)  # flatten generated activations into a matrix of spatial positions by channels.
    style_features = style.reshape(h * w, c)  # flatten style activations into the same matrix layout.
    gram_generated = generated_features.T @ generated_features  # compute generated channel correlations.
    gram_style = style_features.T @ style_features  # compute target style channel correlations.
    denominator = float((2 * h * w * c) ** 2)  # compute the style-loss normalization constant.
    style_grad_flat = (4.0 / denominator) * generated_features @ (gram_generated - gram_style)  # differentiate the squared Gram loss by hand.
    style_grad = style_grad_flat.reshape(h, w, c)  # restore the gradient to activation-tensor shape.
    return alpha * content_grad + beta * style_grad  # return the gradient of the weighted total objective.

def normalize_rows(x):  # normalize each embedding vector to unit length when cosine-like geometry is desired.
    norms = np.linalg.norm(x, axis=1, keepdims=True)  # compute one vector norm per row while keeping matrix shape.
    return x / np.maximum(norms, 1e-12)  # divide safely so no row creates a zero-division error.

def make_identity_embeddings(num_identities=5, photos_per_identity=4, dim=8, spread=0.18):  # synthesize face embeddings grouped by identity.
    centers = normalize_rows(RNG.normal(size=(num_identities, dim)))  # create one normalized latent center per identity.
    embeddings = []  # prepare a list that will hold every noisy embedding.
    labels = []  # prepare a list that will hold the identity label for every embedding.
    for identity in range(num_identities):  # loop over identities so each person gets clustered embeddings.
        for _ in range(photos_per_identity):  # generate several photos per identity.
            sample = centers[identity] + RNG.normal(scale=spread, size=dim)  # add small photo-level noise around the identity center.
            embeddings.append(sample)  # store the noisy embedding vector.
            labels.append(identity)  # store the matching identity label.
    return np.array(embeddings), np.array(labels), centers  # return embeddings, labels, and clean centers.

def pairwise_l2(A, B):  # compute all Euclidean distances between rows of A and rows of B.
    diff = A[:, None, :] - B[None, :, :]  # broadcast row differences into a three-dimensional array.
    return np.sqrt(np.sum(diff ** 2, axis=2))  # collapse feature differences into pairwise Euclidean distances.

def make_toy_image(kind="content", size=16):  # create tiny synthetic images used as activation-like arrays.
    grid_y, grid_x = np.mgrid[0:size, 0:size]  # build coordinate grids for procedural patterns.
    if kind == "content":  # create a structured content pattern with a bright central object.
        image = np.exp(-((grid_x - size * 0.5) ** 2 + (grid_y - size * 0.45) ** 2) / (2 * (size * 0.18) ** 2))  # draw a smooth blob.
        image += 0.35 * (np.abs(grid_x - grid_y) < 2)  # add a diagonal edge so content has recognizable geometry.
    elif kind == "mosaic":  # create a blocky style pattern with repeated color-like channels.
        image = ((grid_x // 4 + grid_y // 4) % 2).astype(float)  # alternate square tiles to mimic a mosaic texture.
        image += 0.25 * np.sin(grid_x * 1.7)  # add high-frequency variation inside the tiles.
    elif kind == "swirl":  # create a smooth swirling style pattern.
        radius = np.sqrt((grid_x - size / 2) ** 2 + (grid_y - size / 2) ** 2)  # compute distance from the image center.
        angle = np.arctan2(grid_y - size / 2, grid_x - size / 2)  # compute angle around the image center.
        image = 0.5 + 0.5 * np.sin(2.5 * angle + 0.8 * radius)  # mix angle and radius into a swirl texture.
    elif kind == "noise":  # create a noisy style edge case.
        image = RNG.normal(loc=0.5, scale=0.35, size=(size, size))  # sample random pixel intensities to mimic noisy texture.
    else:  # reject unknown toy-image names clearly.
        raise ValueError("kind must be content, mosaic, swirl, or noise")  # explain the allowed choices.
    image = np.clip(image, 0.0, 1.0)  # keep the toy image inside displayable intensity bounds.
    channel_1 = image  # use the base pattern as the first channel.
    channel_2 = np.roll(image, shift=2, axis=0)  # create a shifted second channel so Gram correlations are nontrivial.
    channel_3 = np.roll(image, shift=2, axis=1)  # create a shifted third channel for richer style statistics.
    return np.stack([channel_1, channel_2, channel_3], axis=2)  # return a height by width by channel tensor.

def show_image_tensor(tensor, title="image", ax=None):  # display a three-channel toy tensor as an RGB-like image.
    ax = plt.gca() if ax is None else ax  # use current axes when none are supplied.
    clipped = np.clip(tensor, 0.0, 1.0)  # clip values so imshow receives valid image intensities.
    ax.imshow(clipped)  # draw the toy tensor as an image.
    ax.set_title(title)  # label the panel so comparisons are interpretable.
    ax.axis("off")  # remove axes because spatial coordinates are not the lesson focus.
    return ax  # return axes for optional downstream annotations.

### Data — swappable sources

Use the toggles below to choose synthetic face embeddings and toy feature maps. The notebook intentionally includes a clean case and a hard case, because real systems must handle threshold ambiguity, look-alikes, and style artifacts.

In [ ]:
FACE_DATA_SOURCE = "clean"  # choose "clean" for separated identities or "hard" for look-alike/occlusion-style overlap.
STYLE_DATA_SOURCE = "mosaic"  # choose "mosaic", "swirl", or "noise" for the target style texture.
EMBEDDING_DIM = 8  # use small embeddings so every distance calculation remains inspectable.

def load_face_data(source="clean"):  # create a reusable synthetic face-embedding dataset.
    if source == "clean":  # build an easier dataset with compact same-person clusters.
        embeddings, labels, centers = make_identity_embeddings(num_identities=5, photos_per_identity=5, dim=EMBEDDING_DIM, spread=0.14)  # sample clean identity clusters.
    elif source == "hard":  # build a harder dataset with two look-alike identities and more noise.
        embeddings, labels, centers = make_identity_embeddings(num_identities=5, photos_per_identity=5, dim=EMBEDDING_DIM, spread=0.22)  # sample noisier identity clusters.
        centers[1] = normalize_rows((centers[0] + 0.20 * RNG.normal(size=(1, EMBEDDING_DIM))))[0]  # move identity 1 near identity 0 to mimic a look-alike.
        mask = labels == 1  # locate embeddings belonging to the look-alike identity.
        embeddings[mask] = centers[1] + RNG.normal(scale=0.24, size=(mask.sum(), EMBEDDING_DIM))  # regenerate those embeddings around the nearby center.
        embeddings[labels == 3] += RNG.normal(scale=0.35, size=(np.sum(labels == 3), EMBEDDING_DIM))  # perturb one identity to mimic lighting or occlusion.
    else:  # reject invalid face source names.
        raise ValueError("FACE_DATA_SOURCE must be clean or hard")  # give a precise error message for the toggle.
    return embeddings, labels, centers  # return the synthetic face dataset.

face_embeddings, face_labels, face_centers = load_face_data(FACE_DATA_SOURCE)  # load the selected face dataset.
content_tensor = make_toy_image("content", size=16)  # create the content activation tensor.
style_tensor = make_toy_image(STYLE_DATA_SOURCE, size=16)  # create the selected style activation tensor.

print(f"Face data source: {FACE_DATA_SOURCE}")  # report the face-data choice.
print(f"Face embeddings shape: {face_embeddings.shape}")  # report number of images and embedding dimension.
print(f"Style data source: {STYLE_DATA_SOURCE}")  # report the style-data choice.
print(f"Toy tensor shape: {content_tensor.shape}")  # report height, width, and channels for the feature maps.

In [ ]:
plt.figure(figsize=(6, 4))  # create a compact figure for an embedding preview.
projector = RNG.normal(size=(EMBEDDING_DIM, 2))  # create a fixed random projection from 8-D embeddings to 2-D.
projected_embeddings = face_embeddings @ projector  # project embeddings for visualization only.
for identity in np.unique(face_labels):  # draw each identity group separately.
    mask = face_labels == identity  # select embeddings for the current identity.
    plt.scatter(projected_embeddings[mask, 0], projected_embeddings[mask, 1], s=55, color=COLORS[identity], label=f"id {identity}")  # show the projected identity cluster.
plt.title("Synthetic face embeddings projected to 2-D")  # title the diagnostic plot.
plt.xlabel("random projection 1")  # label the horizontal projection axis.
plt.ylabel("random projection 2")  # label the vertical projection axis.
plt.legend(ncol=3, fontsize=8)  # show identity colors without taking too much space.
plt.show()  # render the embedding preview.

▶ What you'll see: each color is one synthetic identity. In `clean`, clusters are better separated; in `hard`, some clusters overlap, foreshadowing false accepts and false rejects.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))  # create side-by-side panels for content and style tensors.
show_image_tensor(content_tensor, "content tensor C", ax=axes[0])  # display the synthetic content tensor.
show_image_tensor(style_tensor, f"style tensor S: {STYLE_DATA_SOURCE}", ax=axes[1])  # display the selected style tensor.
plt.tight_layout()  # reduce whitespace between the two panels.
plt.show()  # render the toy content and style inputs.

▶ What you'll see: the content tensor has a simple object-like blob and edge; the style tensor has texture statistics such as blocks, swirls, or random high-frequency noise.

### 📖 Concept walkthrough — build each idea from scratch

Before the warm-up examples, we build the face-recognition and style-transfer ideas from scratch, one small step at a time. Everything here uses only NumPy + Matplotlib and tiny inline vectors or arrays, so every distance, loss, and matrix is inspectable. Variables carry a `_w` suffix so they never collide with the examples below.

In [ ]:
import numpy as np  # NumPy gives us tiny embedding vectors, distances, feature maps, and Gram matrices.
import matplotlib.pyplot as plt  # Matplotlib lets us check verification thresholds, nearest matches, triplet distances, and style correlations visually.
np.random.seed(0)  # fix the seed so every printed value and plot below is reproducible.

#### 1. Face verification vs recognition: one-to-one check vs one-to-many search

Face verification asks, "Is this query the claimed identity?" and answers with one distance compared to a threshold. Face recognition asks, "Who is this query among all enrolled identities?" and answers by searching the whole gallery for the nearest embedding. We build both from the same tiny vectors because the modeling primitive is identical — compare embeddings by distance — but the decision question is different.

In [ ]:
alice_ref_w = np.array([0.10, 0.20, 0.90])  # store Alice's enrolled face embedding as a small numeric vector.
alice_query_w = np.array([0.12, 0.18, 0.88])  # store a query embedding that should look like Alice.
bob_ref_w = np.array([0.80, 0.10, 0.20])  # store Bob's enrolled face embedding for the gallery.
chen_ref_w = np.array([0.20, 0.85, 0.10])  # store Chen's enrolled face embedding for the gallery.
threshold_w = 0.12  # choose the maximum distance allowed for accepting a 1:1 verification claim.
verify_distance_w = np.linalg.norm(alice_query_w - alice_ref_w)  # compute the query-to-claimed-reference distance.
verify_accept_w = verify_distance_w < threshold_w  # accept only when the distance is below the threshold.

print("verification distance:", round(verify_distance_w, 3))  # inspect the single 1:1 distance.
print("accept Alice claim?", verify_accept_w)  # inspect the verification decision.

▶ What you'll see: the query is close to Alice's reference, so the 1:1 verification claim is accepted.

In [ ]:
gallery_names_w = np.array(["Alice", "Bob", "Chen"])  # name the enrolled gallery identities.
gallery_embeddings_w = np.vstack([alice_ref_w, bob_ref_w, chen_ref_w])  # stack references into one 1:N gallery matrix.
recognition_distances_w = np.linalg.norm(gallery_embeddings_w - alice_query_w, axis=1)  # compute one query-to-gallery distance per identity.
best_index_w = int(np.argmin(recognition_distances_w))  # find the nearest gallery embedding.
best_name_w = gallery_names_w[best_index_w]  # look up the nearest identity name.

print("recognition distances:", dict(zip(gallery_names_w, np.round(recognition_distances_w, 3))))  # inspect every 1:N comparison.
print("recognized as:", best_name_w)  # inspect the nearest-neighbor identity.

▶ What you'll see: recognition compares the same query to every enrolled person and chooses Alice as the nearest match.

In [ ]:
plt.figure(figsize=(5.6, 3.4))  # create a compact bar chart for the recognition distances.
plt.bar(gallery_names_w, recognition_distances_w, color=["tab:green", "tab:blue", "tab:orange"])  # draw one distance bar per gallery identity.
plt.axhline(threshold_w, color="crimson", linestyle="--", label="verification threshold")  # draw the 1:1 acceptance cutoff for comparison.
plt.scatter([best_name_w], [recognition_distances_w[best_index_w]], color="black", zorder=3, label="nearest match")  # mark the winning gallery identity.
plt.ylabel("Euclidean embedding distance")  # label the distance scale used by both tasks.
plt.legend(loc="best")  # show what the threshold and marker mean.
plt.title("1: verification threshold and recognition distances")  # title the concept plot.
plt.show()  # render the distance comparison.

▶ What you'll see: Alice's bar is both the nearest recognition match and below the verification threshold.

*Why it's done this way: verification and recognition reuse the same embedding distance, but verification is a yes/no thresholded 1:1 test while recognition is a nearest-neighbor 1:N search. Separating the two tasks makes it clear why a system can accept a claimed identity, search a gallery, or reject if all distances are too large.*

#### 2. Siamese embeddings and one-shot learning: recognize a new identity from one example

A Siamese model runs the same encoder on both images, so the output vectors live in one shared space where distance is meaningful. That matters for one-shot learning because a new person does not need a freshly trained classifier head; one enrolled embedding can be compared to a query immediately. We simulate the encoder's output directly as vectors and recognize the query by nearest embedding.

In [ ]:
one_shot_names_w = np.array(["Alice", "Bob", "Dora (one shot)"])  # include a brand-new identity with only one stored example.
one_shot_gallery_w = np.array([[0.10, 0.20, 0.90], [0.80, 0.10, 0.20], [0.42, 0.72, 0.35]])  # store one embedding per identity.
one_shot_query_w = np.array([0.39, 0.75, 0.33])  # create a query embedding from the new identity Dora.

print("one-shot gallery:\n", one_shot_gallery_w)  # inspect the tiny gallery matrix.
print("query embedding:", one_shot_query_w)  # inspect the query vector before comparing distances.

▶ What you'll see: Dora has only one reference vector, but it is already enough to enter the gallery.

In [ ]:
one_shot_distances_w = np.linalg.norm(one_shot_gallery_w - one_shot_query_w, axis=1)  # compare the query with each stored embedding.
one_shot_index_w = int(np.argmin(one_shot_distances_w))  # choose the nearest embedding as the predicted identity.
one_shot_prediction_w = one_shot_names_w[one_shot_index_w]  # translate the nearest index into a name.

print("one-shot distances:", dict(zip(one_shot_names_w, np.round(one_shot_distances_w, 3))))  # inspect all comparison scores.
print("predicted identity:", one_shot_prediction_w)  # inspect the one-shot recognition result.

The reason this works is that the model has learned a reusable similarity function $d(x,z)=\lVert f(x)-f(z)\rVert_2$, not a fixed list of class weights. Once embeddings are meaningful, adding a class means adding an example to the gallery.
▶ What you'll see: the query is closest to Dora's single enrolled example, so the new identity is recognized without retraining.

In [ ]:
plt.figure(figsize=(5.6, 3.4))  # create a bar chart focused on one-shot nearest-neighbor matching.
plt.bar(one_shot_names_w, one_shot_distances_w, color=["tab:blue", "tab:orange", "tab:green"])  # draw query-to-gallery distances.
plt.scatter([one_shot_prediction_w], [one_shot_distances_w[one_shot_index_w]], color="black", zorder=3, label="nearest")  # mark the winning one-shot example.
plt.ylabel("distance to query")  # label the vertical distance axis.
plt.xticks(rotation=12)  # rotate the long one-shot label slightly so it remains readable.
plt.legend(loc="best")  # show the nearest-match marker.
plt.title("2: one-shot recognition by embedding distance")  # title the concept plot.
plt.show()  # render the one-shot distance chart.

▶ What you'll see: Dora's single reference has the shortest bar, illustrating one-shot recognition.

In [ ]:
plt.figure(figsize=(5.2, 4.0))  # create a small two-coordinate embedding sketch.
plt.scatter(one_shot_gallery_w[:, 0], one_shot_gallery_w[:, 1], s=90, color=["tab:blue", "tab:orange", "tab:green"], label="gallery")  # draw gallery embeddings using the first two coordinates.
plt.scatter(one_shot_query_w[0], one_shot_query_w[1], s=140, marker="*", color="black", label="query")  # draw the query embedding as a star.
for i_w, name_w in enumerate(one_shot_names_w):  # annotate each gallery point with its identity name.
    plt.text(one_shot_gallery_w[i_w, 0] + 0.01, one_shot_gallery_w[i_w, 1] + 0.01, name_w, fontsize=8)  # place a readable label next to the point.
plt.plot([one_shot_query_w[0], one_shot_gallery_w[one_shot_index_w, 0]], [one_shot_query_w[1], one_shot_gallery_w[one_shot_index_w, 1]], color="black", linestyle="--")  # connect the query to its nearest reference.
plt.xlabel("embedding coordinate 1")  # label the horizontal embedding coordinate.
plt.ylabel("embedding coordinate 2")  # label the vertical embedding coordinate.
plt.legend(loc="best")  # show which marker is the query.
plt.title("2: query lands near the one-shot example")  # title the geometry plot.
plt.show()  # render the embedding sketch.

▶ What you'll see: the query star sits nearest to Dora's lone gallery point in embedding space.

*Why it's done this way: Siamese embeddings turn identity prediction into metric lookup, so a new identity can be enrolled by storing one vector. The shared encoder supplies the reusable geometry, and nearest-neighbor comparison supplies the one-shot decision.*

#### 3. Triplet loss: make positives closer than negatives by a margin

Triplet loss trains on an anchor $a$, a positive $p$ from the same identity, and a negative $n$ from a different identity. The loss is

$$
\max\left(0, \lVert a-p\rVert^2 - \lVert a-n\rVert^2 + \alpha\right)
$$

It is zero only when $\lVert a-p\rVert^2 + \alpha \le \lVert a-n\rVert^2$, so the negative must be farther away by at least the margin. We compute the loss, then take one hand-built step that pulls the positive toward the anchor and pushes the negative away.

In [ ]:
anchor_w = np.array([0.20, 0.30])  # create a 2-D anchor embedding for one identity.
positive_w = np.array([0.55, 0.48])  # create a same-identity positive that is too far from the anchor.
negative_w = np.array([0.65, 0.35])  # create a different-identity negative that is too close to the anchor.
margin_w = 0.25  # require the negative squared distance to exceed the positive squared distance by this amount.
d_ap_before_w = np.sum((anchor_w - positive_w) ** 2)  # compute squared anchor-positive distance before the update.
d_an_before_w = np.sum((anchor_w - negative_w) ** 2)  # compute squared anchor-negative distance before the update.
loss_before_w = max(0.0, d_ap_before_w - d_an_before_w + margin_w)  # compute triplet loss before the update.

print("before d(a,p)^2:", round(d_ap_before_w, 3))  # inspect positive distance before training.
print("before d(a,n)^2:", round(d_an_before_w, 3))  # inspect negative distance before training.
print("before triplet loss:", round(loss_before_w, 3))  # inspect whether the margin is violated.

▶ What you'll see: the negative is not far enough beyond the positive, so the triplet loss is positive.

In [ ]:
learning_rate_w = 0.45  # choose a visible step size for this toy update.
positive_after_w = positive_w - learning_rate_w * 2.0 * (positive_w - anchor_w)  # pull the positive toward the anchor.
negative_after_w = negative_w + learning_rate_w * 2.0 * (negative_w - anchor_w)  # push the negative farther from the anchor.
d_ap_after_w = np.sum((anchor_w - positive_after_w) ** 2)  # recompute squared anchor-positive distance after the update.
d_an_after_w = np.sum((anchor_w - negative_after_w) ** 2)  # recompute squared anchor-negative distance after the update.
loss_after_w = max(0.0, d_ap_after_w - d_an_after_w + margin_w)  # recompute triplet loss after the update.

print("after d(a,p)^2:", round(d_ap_after_w, 3))  # inspect the reduced positive distance.
print("after d(a,n)^2:", round(d_an_after_w, 3))  # inspect the enlarged negative distance.
print("after triplet loss:", round(loss_after_w, 3))  # inspect how the margin violation changed.

The margin prevents the model from settling for "positive is just barely closer." It creates a safety buffer, so embeddings stay useful even when new images are noisy or identities look similar.
▶ What you'll see: the positive moves closer, the negative moves farther, and the loss drops.

In [ ]:
dist_before_w = np.array([d_ap_before_w, d_an_before_w])  # collect before-update squared distances for plotting.
dist_after_w = np.array([d_ap_after_w, d_an_after_w])  # collect after-update squared distances for plotting.
x_triplet_w = np.arange(2)  # create two bar positions for positive and negative distances.
plt.figure(figsize=(5.8, 3.6))  # create a compact before/after distance chart.
plt.bar(x_triplet_w - 0.18, dist_before_w, width=0.36, label="before", color="lightgray")  # draw distances before the update.
plt.bar(x_triplet_w + 0.18, dist_after_w, width=0.36, label="after", color="tab:purple")  # draw distances after the update.
plt.axhline(d_ap_after_w + margin_w, color="crimson", linestyle="--", label="after positive + margin")  # show the required negative-distance target after the update.
plt.xticks(x_triplet_w, ["$\\lVert a-p\\rVert^2$", "$\\lVert a-n\\rVert^2$"])  # label the two squared distances.
plt.ylabel("squared distance")  # label the vertical distance scale.
plt.legend(loc="best")  # show before/after and margin meanings.
plt.title("3: triplet update separates positive and negative")  # title the triplet-loss plot.
plt.show()  # render the before/after comparison.

▶ What you'll see: the positive-distance bar shrinks, the negative-distance bar grows, and the margin condition becomes easier to satisfy.

In [ ]:
plt.figure(figsize=(5.0, 4.0))  # create a 2-D embedding movement plot.
plt.scatter(anchor_w[0], anchor_w[1], s=130, color="black", label="anchor")  # draw the fixed anchor point.
plt.scatter(positive_w[0], positive_w[1], s=90, color="tab:green", alpha=0.45, label="positive before")  # draw the original positive point.
plt.scatter(negative_w[0], negative_w[1], s=90, color="tab:red", alpha=0.45, label="negative before")  # draw the original negative point.
plt.scatter(positive_after_w[0], positive_after_w[1], s=120, marker="*", color="tab:green", label="positive after")  # draw the moved positive point.
plt.scatter(negative_after_w[0], negative_after_w[1], s=120, marker="*", color="tab:red", label="negative after")  # draw the moved negative point.
plt.plot([positive_w[0], positive_after_w[0]], [positive_w[1], positive_after_w[1]], color="tab:green", linestyle="--")  # show the positive moving toward the anchor.
plt.plot([negative_w[0], negative_after_w[0]], [negative_w[1], negative_after_w[1]], color="tab:red", linestyle="--")  # show the negative moving away from the anchor.
plt.xlabel("embedding coordinate 1")  # label the horizontal embedding coordinate.
plt.ylabel("embedding coordinate 2")  # label the vertical embedding coordinate.
plt.legend(loc="best", fontsize=8)  # keep all point meanings visible.
plt.title("3: embedding motion from triplet loss")  # title the motion plot.
plt.show()  # render the triplet geometry.

▶ What you'll see: the positive star moves toward the anchor while the negative star moves away.

*Why it's done this way: triplet loss trains the embedding space by relative comparisons instead of fixed class labels. The margin $\alpha$ forces a useful buffer, so same-identity pairs become not just closer than different-identity pairs, but closer by enough to be reliable.*

#### 4. Neural style transfer: content loss and Gram-matrix style loss

Neural style transfer does not compare final class labels; it compares internal activations. Content loss keeps the generated activation map spatially close to the content activation map, while style loss compares Gram matrices of channel co-activations. We use tiny feature maps because the core math is just squared differences and $G=F^\top F$ for flattened features $F$.

$$
J_{\text{content}}=\frac{1}{2}\sum_{i,j,k}\left(a^{(G)}_{ijk}-a^{(C)}_{ijk}\right)^2
$$

$$
J_{\text{style}}=\frac{1}{(2n_Hn_Wn_C)^2}\sum_{k,k'}\left(G^{(S)}_{kk'}-G^{(G)}_{kk'}\right)^2
$$

In [ ]:
content_features_w = np.array([[[1.0, 0.0], [0.8, 0.1]], [[0.2, 0.7], [0.0, 1.0]]])  # create a 2x2x2 content activation map with spatial structure.
style_features_w = np.array([[[1.0, 1.0], [0.9, 0.8]], [[0.2, 0.1], [0.1, 0.2]]])  # create a style activation map with strong channel co-activation in the top row.
generated_features_w = np.array([[[0.7, 0.2], [0.6, 0.2]], [[0.2, 0.5], [0.1, 0.8]]])  # create a generated activation map that partially matches both targets.

print("content shape:", content_features_w.shape)  # inspect the height, width, and channel count.
print("style shape:", style_features_w.shape)  # inspect that style uses the same activation shape.
print("generated shape:", generated_features_w.shape)  # inspect that generated activations are comparable.

▶ What you'll see: all three toy feature maps have shape 2 by 2 by 2, so we can compare them directly.

In [ ]:
content_difference_w = generated_features_w - content_features_w  # subtract activations at matching spatial locations and channels.
content_loss_w = 0.5 * np.sum(content_difference_w ** 2)  # compute one-half squared content loss.

print("content difference tensor:\n", np.round(content_difference_w, 2))  # inspect where generated content differs from target content.
print("content loss:", round(float(content_loss_w), 4))  # inspect the scalar content penalty.

Content loss preserves layout because it compares activation values at the same spatial positions. If a generated feature moves to the wrong location, the squared difference grows even if the channels are similar overall.
▶ What you'll see: the tensor shows location-by-location mismatches, and the scalar summarizes their squared size.

In [ ]:
def gram_style_w(activation_w):  # define a Gram helper for activations shaped height by width by channels.
    h_w, width_w, channels_w = activation_w.shape  # read the spatial and channel dimensions.
    features_w = activation_w.reshape(h_w * width_w, channels_w)  # flatten spatial locations into rows while keeping channels as columns.
    return features_w.T @ features_w  # compute G = F^T F, the channel-by-channel co-activation matrix.
gram_style_target_w = gram_style_w(style_features_w)  # compute the target style Gram matrix.
gram_generated_w = gram_style_w(generated_features_w)  # compute the generated Gram matrix.
style_loss_w = np.sum((gram_style_target_w - gram_generated_w) ** 2) / float((2 * 2 * 2 * 2) ** 2)  # compute a small normalized Gram loss.

print("style Gram matrix:\n", np.round(gram_style_target_w, 3))  # inspect target channel correlations.
print("generated Gram matrix:\n", np.round(gram_generated_w, 3))  # inspect generated channel correlations.
print("style loss:", round(float(style_loss_w), 5))  # inspect the scalar style penalty.

The Gram matrix captures style because each entry sums how often two channels activate together across all spatial positions. Since the spatial rows are summed away, it keeps texture and co-activation statistics while mostly ignoring exact layout.
▶ What you'll see: two 2x2 Gram matrices whose differences become the style loss.

In [ ]:
fig_style_w, axes_style_w = plt.subplots(1, 2, figsize=(6.4, 3.0))  # create side-by-side Gram-matrix panels.
image0_style_w = axes_style_w[0].imshow(gram_style_target_w, cmap="magma")  # display the target style Gram matrix.
axes_style_w[0].set_title("4: style Gram target")  # title the target Gram panel.
axes_style_w[0].set_xlabel("channel")  # label the target panel x-axis as channels.
axes_style_w[0].set_ylabel("channel")  # label the target panel y-axis as channels.
image1_style_w = axes_style_w[1].imshow(gram_generated_w, cmap="magma")  # display the generated Gram matrix.
axes_style_w[1].set_title("4: generated Gram")  # title the generated Gram panel.
axes_style_w[1].set_xlabel("channel")  # label the generated panel x-axis as channels.
axes_style_w[1].set_ylabel("channel")  # label the generated panel y-axis as channels.
plt.colorbar(image0_style_w, ax=axes_style_w[0], fraction=0.046)  # add a color scale for the target correlations.
plt.colorbar(image1_style_w, ax=axes_style_w[1], fraction=0.046)  # add a color scale for the generated correlations.
plt.tight_layout()  # reduce overlap between panels and colorbars.
plt.show()  # render the Gram-matrix comparison.

▶ What you'll see: brighter cells indicate stronger channel co-activations, the toy stand-in for visual texture.

In [ ]:
alpha_w = 1.0  # choose the content weight in the total style-transfer objective.
beta_w = 8.0  # choose the style weight so the small normalized style loss still matters.
total_loss_w = alpha_w * content_loss_w + beta_w * style_loss_w  # combine content and style losses into one objective.

print("weighted content term:", round(float(alpha_w * content_loss_w), 4))  # inspect the contribution from content preservation.
print("weighted style term:", round(float(beta_w * style_loss_w), 4))  # inspect the contribution from style matching.
print("total loss:", round(float(total_loss_w), 4))  # inspect the final generated-image objective.

▶ What you'll see: $\alpha$ and $\beta$ decide how much the generated activation prioritizes layout versus texture statistics.

*Why it's done this way: content loss compares feature maps in place to preserve object layout, while Gram-matrix style loss compares channel correlations to preserve texture without requiring the same spatial arrangement. The weighted objective lets neural style transfer trade off recognizable content against the desired style.*

### 🟢 Basics (warm-up)

#### B1. Cosine similarity between two face embeddings

**Goal.** Compare two toy 3-D embeddings by direction, not length. We'll build this in **2 steps**.

In [ ]:
embedding_a_b1 = np.array([1.0, 2.0, 2.0])  # create the first toy face embedding.
embedding_b_b1 = np.array([2.0, 1.0, 2.0])  # create the second toy face embedding.
similarity_b1 = cosine_similarity(embedding_a_b1, embedding_b_b1)  # compute scale-free directional similarity.
angle_b1 = degrees(acos(np.clip(similarity_b1, -1.0, 1.0)))  # convert similarity into an angle for intuition.

print("Embedding A:", embedding_a_b1)  # display the first embedding.
print("Embedding B:", embedding_b_b1)  # display the second embedding.
print(f"Cosine similarity = {similarity_b1:.3f}")  # display the cosine similarity.
print(f"Angle between embeddings = {angle_b1:.1f} degrees")  # display the geometric angle.

In [ ]:
plt.figure(figsize=(4.5, 4.5))  # create a square plot for a tiny vector sketch.
plt.quiver([0, 0], [0, 0], [embedding_a_b1[0], embedding_b_b1[0]], [embedding_a_b1[1], embedding_b_b1[1]], angles="xy", scale_units="xy", scale=1, color=[COLORS[0], COLORS[1]])  # draw the first two coordinates as arrows.
plt.xlim(0, 2.5)  # set horizontal limits so both arrows fit.
plt.ylim(0, 2.5)  # set vertical limits so both arrows fit.
plt.xlabel("embedding coordinate 1")  # label the first coordinate.
plt.ylabel("embedding coordinate 2")  # label the second coordinate.
plt.title("B1 angle sketch using first two coordinates")  # title the vector sketch.
plt.grid(True)  # keep the geometric grid visible.
plt.show()  # render the angle sketch.

▶ What you'll see: two arrows pointing in similar directions, matching a high cosine similarity. The sketch uses only two coordinates, while the printed value uses all three.

#### B2. L2 distance and threshold decision for one face pair

**Goal.** Turn one embedding distance into an accept/reject verification decision. We'll build this in **2 steps**.

In [ ]:
reference_b2 = np.array([0.20, 0.10, 0.75, -0.15])  # create a toy enrolled reference embedding.
query_b2 = np.array([0.24, 0.18, 0.68, -0.11])  # create a toy query embedding that should be close.
threshold_b2 = 0.16  # set a fixed verification threshold.
distance_b2 = l2_distance(reference_b2, query_b2)  # compute Euclidean distance between reference and query.
decision_b2 = "ACCEPT: same claimed identity" if distance_b2 <= threshold_b2 else "REJECT: different identity"  # compare distance to threshold.

print(f"L2 distance = {distance_b2:.3f}")  # display the measured distance.
print(f"Threshold = {threshold_b2:.3f}")  # display the decision boundary.
print(decision_b2)  # display the final verification decision.

In [ ]:
plt.figure(figsize=(5.5, 1.8))  # create a horizontal distance-gauge figure.
plt.axvline(threshold_b2, color="black", linestyle="--", label="threshold")  # draw the accept/reject threshold.
plt.scatter([distance_b2], [0], s=160, color=COLORS[2], label="pair distance")  # draw the observed pair distance.
plt.yticks([])  # hide the meaningless vertical axis.
plt.xlim(0, 0.35)  # show a small distance range around the threshold.
plt.xlabel("embedding L2 distance")  # label the distance axis.
plt.title("B2 verification decision")  # title the decision plot.
plt.legend()  # show which marker is the distance and which line is the threshold.
plt.show()  # render the distance-gauge plot.

▶ What you'll see: the distance marker falls to the left of the threshold line, so the pair is accepted as the same claimed identity.

#### B3. Gram matrix of a 2×2 two-channel activation map

**Goal.** Compute a tiny style matrix that measures channel correlations. We'll build this in **3 steps**.

In [ ]:
activation_b3 = np.array([[[1.0, 0.0], [2.0, 1.0]], [[0.0, 1.0], [1.0, 2.0]]])  # create a 2 by 2 activation map with 2 channels.
features_b3 = activation_b3.reshape(4, 2)  # flatten the four spatial locations into rows.
gram_b3 = gram_matrix(activation_b3)  # compute the channel-by-channel Gram matrix.

print("Flattened spatial features:")  # introduce the flattened representation.
print(features_b3)  # show every spatial location's two channel values.
print("Gram matrix:")  # introduce the Gram result.
print(gram_b3)  # show channel self-correlations and cross-correlations.

In [ ]:
plt.figure(figsize=(4, 3.5))  # create a small heatmap figure.
plt.imshow(gram_b3, cmap="Blues")  # display the Gram matrix as a heatmap.
plt.colorbar(label="channel correlation total")  # add a colorbar so values are readable.
plt.xticks([0, 1], ["channel 0", "channel 1"])  # label Gram columns by channel.
plt.yticks([0, 1], ["channel 0", "channel 1"])  # label Gram rows by channel.
plt.title("B3 Gram matrix heatmap")  # title the tiny style matrix plot.
for i in range(2):  # loop over Gram rows for numeric annotations.
    for j in range(2):  # loop over Gram columns for numeric annotations.
        plt.text(j, i, f"{gram_b3[i, j]:.0f}", ha="center", va="center", color="black")  # write each Gram entry on the heatmap.
plt.tight_layout()  # keep labels from overlapping.
plt.show()  # render the Gram heatmap.

▶ What you'll see: diagonal entries measure each channel's total energy, while off-diagonal entries measure how often the two channels are active together.


#### B4. Normalize one face embedding to unit length

**Goal.** Convert one embedding into a unit vector before cosine-style comparisons. We'll build this in **2 steps**.

In [ ]:
embedding_b4 = np.array([3.0, 4.0, 0.0])  # create a toy face embedding with length five.
norm_b4 = np.linalg.norm(embedding_b4)  # measure the vector length before normalization.
unit_b4 = embedding_b4 / norm_b4  # divide by the norm so the vector length becomes one.

print("original embedding:", embedding_b4)  # show the raw vector.
print(f"original norm = {norm_b4:.3f}")  # show the raw vector length.
print("normalized embedding:", unit_b4)  # show the unit-length vector.
print(f"normalized norm = {np.linalg.norm(unit_b4):.3f}")  # verify the new length.

In [ ]:
plt.figure(figsize=(4.5, 4.5))  # create a small vector plot.
plt.quiver([0, 0], [0, 0], [embedding_b4[0], unit_b4[0]], [embedding_b4[1], unit_b4[1]], angles="xy", scale_units="xy", scale=1, color=[COLORS[0], COLORS[3]])  # draw raw and normalized directions.
plt.text(embedding_b4[0] + 0.05, embedding_b4[1], "raw", color=COLORS[0])  # label the raw vector.
plt.text(unit_b4[0] + 0.05, unit_b4[1], "unit", color=COLORS[3])  # label the normalized vector.
plt.xlim(0, 3.5)  # set x-limits for both arrows.
plt.ylim(0, 4.5)  # set y-limits for both arrows.
plt.title("B4 normalization keeps direction")  # title the normalization sketch.
plt.grid(True)  # keep the coordinate grid visible.
plt.show()  # render the plot.

▶ What you'll see: the arrow becomes shorter, but it points in the same direction.

👀 **Takeaway.** Normalization removes scale so comparisons focus on direction.

#### B5. Content loss between two tiny feature maps

**Goal.** Measure how far generated activations are from content activations. We'll build this in **2 steps**.

In [ ]:
content_b5 = np.array([[[0.0], [1.0]], [[1.0], [0.0]]])  # create a tiny content feature map.
generated_b5 = np.array([[[0.2], [0.7]], [[0.8], [0.1]]])  # create a slightly changed generated feature map.
loss_b5 = content_loss(content_b5, generated_b5)  # compute one-half squared feature difference.

print("content feature map:\n", content_b5[:, :, 0])  # print the content activations.
print("generated feature map:\n", generated_b5[:, :, 0])  # print the generated activations.
print(f"content loss = {loss_b5:.4f}")  # print the scalar content penalty.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))  # create panels for content, generated, and absolute error.
for ax, matrix, title in zip(axes, [content_b5[:, :, 0], generated_b5[:, :, 0], np.abs(content_b5[:, :, 0] - generated_b5[:, :, 0])], ["content", "generated", "absolute difference"]):  # loop over the maps.
    im = ax.imshow(matrix, cmap="viridis", vmin=0.0, vmax=1.0)  # display each map on a shared scale.
    ax.set_title(title)  # label the panel.
    ax.axis("off")  # remove ticks.
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75)  # add one shared colorbar.
plt.show()  # render the comparison.

▶ What you'll see: larger activation differences contribute more to the content loss.

👀 **Takeaway.** Content loss is just MSE-style distance in a chosen feature layer.

#### B6. Style loss between two Gram matrices

**Goal.** Compare texture statistics after converting activations to Gram matrices. We'll build this in **3 steps**.

In [ ]:
style_b6 = np.array([[[1.0, 0.0], [1.0, 1.0]], [[0.0, 1.0], [1.0, 0.0]]])  # create a tiny style activation tensor.
generated_b6 = np.array([[[0.8, 0.2], [1.1, 0.7]], [[0.1, 0.9], [0.9, 0.1]]])  # create a generated tensor with similar texture.
gram_style_b6 = gram_matrix(style_b6)  # compute target style correlations.
gram_generated_b6 = gram_matrix(generated_b6)  # compute generated correlations.
style_mse_b6 = float(np.mean((gram_style_b6 - gram_generated_b6) ** 2))  # compute plain MSE between Gram matrices.

print("style Gram:\n", gram_style_b6)  # print target Gram matrix.
print("generated Gram:\n", gram_generated_b6)  # print generated Gram matrix.
print(f"Gram MSE = {style_mse_b6:.4f}")  # print the style mismatch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3))  # create side-by-side Gram heatmaps.
for ax, matrix, title in zip(axes, [gram_style_b6, gram_generated_b6], ["style Gram", "generated Gram"]):  # loop over Gram matrices.
    im = ax.imshow(matrix, cmap="Blues")  # visualize channel correlations.
    ax.set_title(title)  # label each Gram matrix.
    ax.set_xticks([0, 1])  # label channel columns.
    ax.set_yticks([0, 1])  # label channel rows.
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75)  # add a shared colorbar.
plt.show()  # render the Gram comparison.

▶ What you'll see: the scalar loss is small when channel-correlation patterns are close.

👀 **Takeaway.** Style transfer matches texture by matching Gram matrices, not pixel positions.

#### B7. Find the nearest gallery embedding

**Goal.** Do one tiny one-to-many face-recognition lookup. We'll build this in **3 steps**.

In [ ]:
gallery_b7 = np.array([[0.0, 0.0], [1.0, 0.2], [0.2, 1.0]])  # create three enrolled gallery embeddings.
names_b7 = np.array(["Ana", "Bo", "Cy"])  # name the enrolled identities.
query_b7 = np.array([0.9, 0.1])  # create one query embedding.
distances_b7 = np.linalg.norm(gallery_b7 - query_b7, axis=1)  # compute query-to-gallery L2 distances.
best_b7 = int(np.argmin(distances_b7))  # find the nearest gallery row.
for name, distance in zip(names_b7, distances_b7):  # print each candidate distance.
    print(f"distance to {name}: {distance:.3f}")  # show one lookup score.

print("nearest identity:", names_b7[best_b7])  # print the predicted identity.

In [ ]:
plt.figure(figsize=(5, 4))  # create a tiny embedding map.
plt.scatter(gallery_b7[:, 0], gallery_b7[:, 1], s=100, label="gallery")  # plot enrolled identities.
plt.scatter([query_b7[0]], [query_b7[1]], s=140, marker="*", color="crimson", label="query")  # plot the query.
for name, xy in zip(names_b7, gallery_b7):  # label gallery points.
    plt.text(xy[0] + 0.03, xy[1] + 0.03, name)  # write identity labels.
plt.title("B7 nearest gallery embedding")  # title the lookup plot.
plt.legend()  # identify gallery and query markers.
plt.show()  # render the map.

▶ What you'll see: the query star is closest to Bo, so Bo is returned.

👀 **Takeaway.** Recognition is nearest-neighbor search over enrolled embeddings.

#### B8. Triplet margin score for one anchor-positive-negative set

**Goal.** Compute the raw margin signal $d(a,p)-d(a,n)+\alpha$. We'll build this in **2 steps**.

In [ ]:
anchor_b8 = np.array([0.0, 0.0])  # create an anchor embedding.
positive_b8 = np.array([0.2, 0.1])  # create a nearby same-identity positive.
negative_b8 = np.array([0.7, 0.2])  # create a farther different-identity negative.
margin_b8 = 0.4  # set the required separation margin.
d_ap_b8 = l2_distance(anchor_b8, positive_b8)  # compute anchor-positive distance.
d_an_b8 = l2_distance(anchor_b8, negative_b8)  # compute anchor-negative distance.
raw_b8 = d_ap_b8 - d_an_b8 + margin_b8  # compute the triplet margin expression.
loss_b8 = max(raw_b8, 0.0)  # hinge the raw score at zero.

print(f"d(anchor, positive) = {d_ap_b8:.3f}")  # print same-identity distance.
print(f"d(anchor, negative) = {d_an_b8:.3f}")  # print different-identity distance.
print(f"triplet loss = max({raw_b8:.3f}, 0) = {loss_b8:.3f}")  # print the hinge result.

In [ ]:
plt.figure(figsize=(5, 4))  # create a tiny triplet plot.
plt.scatter(*anchor_b8, s=120, label="anchor")  # draw the anchor.
plt.scatter(*positive_b8, s=120, label="positive")  # draw the positive.
plt.scatter(*negative_b8, s=120, label="negative")  # draw the negative.
plt.plot([anchor_b8[0], positive_b8[0]], [anchor_b8[1], positive_b8[1]], color=COLORS[1])  # connect anchor to positive.
plt.plot([anchor_b8[0], negative_b8[0]], [anchor_b8[1], negative_b8[1]], color=COLORS[2])  # connect anchor to negative.
plt.title("B8 triplet distances")  # title the margin sketch.
plt.legend()  # identify triplet points.
plt.axis("equal")  # preserve distance geometry.
plt.show()  # render the plot.

▶ What you'll see: the loss is zero only if the negative is far enough beyond the positive.

👀 **Takeaway.** Triplet loss trains relative ordering, not an absolute class label.

#### B9. Compare all query-to-gallery distances

**Goal.** Turn one query into a distance vector over all enrolled identities. We'll build this in **2 steps**.

In [ ]:
query_b9 = np.array([[0.25, 0.95]])  # create one query row vector.
gallery_b9 = np.array([[0.1, 0.0], [1.0, 0.2], [0.2, 1.1], [0.8, 0.8]])  # create four gallery embeddings.
names_b9 = ["Ana", "Bo", "Cy", "Di"]  # name the gallery entries.
distances_b9 = pairwise_l2(query_b9, gallery_b9).ravel()  # compute all query-to-gallery distances with the helper.
rank_b9 = np.argsort(distances_b9)  # sort candidate identities from nearest to farthest.
for idx in rank_b9:  # print ranked distances.
    print(f"{names_b9[idx]}: {distances_b9[idx]:.3f}")  # show one candidate.

In [ ]:
plt.figure(figsize=(6, 3))  # create a ranked-distance bar chart.
plt.bar([names_b9[i] for i in rank_b9], distances_b9[rank_b9], color="steelblue")  # plot distances in sorted order.
plt.ylabel("L2 distance to query")  # label the distance axis.
plt.title("B9 ranked gallery distances")  # title the ranking plot.
plt.show()  # render the bars.

▶ What you'll see: the closest bar is the best candidate, but the full ranking shows ambiguity.

👀 **Takeaway.** Recognition systems often inspect top-k distances, not just the winner.

#### B10. Reject a nearest face match with a threshold

**Goal.** Combine nearest-neighbor recognition with an unknown-person rejection rule. We'll build this in **2 steps**.

In [ ]:
threshold_b10 = 0.30  # set the maximum allowed nearest-neighbor distance.
names_b10 = ["Ana", "Bo", "Cy", "Di"]  # name four enrolled gallery entries.
distances_b10 = np.array([0.72, 0.58, 0.34, 0.49])  # create one query's nearest-neighbor distance vector.
best_idx_b10 = int(np.argmin(distances_b10))  # find the nearest candidate.
best_distance_b10 = distances_b10[best_idx_b10]  # read the nearest distance.
result_b10 = names_b10[best_idx_b10] if best_distance_b10 <= threshold_b10 else "unknown"  # reject if even the nearest is too far.

print(f"best candidate = {names_b10[best_idx_b10]}")  # print the nearest gallery name.
print(f"best distance = {best_distance_b10:.3f}")  # print the nearest distance.
print(f"threshold = {threshold_b10:.3f}")  # print the recognition threshold.
print("final decision:", result_b10)  # print either an identity or unknown.

In [ ]:
plt.figure(figsize=(6, 2.4))  # create a compact threshold chart.
plt.axhline(threshold_b10, color="black", linestyle="--", label="reject threshold")  # draw the rejection threshold.
plt.bar(names_b10, distances_b10, color=[COLORS[2] if i == best_idx_b10 else COLORS[0] for i in range(len(names_b10))])  # highlight the nearest candidate.
plt.ylabel("distance")  # label the distance axis.
plt.title("B10 accept nearest only below threshold")  # title the decision plot.
plt.legend()  # label the threshold.
plt.show()  # render the plot.

▶ What you'll see: a nearest identity can still be rejected if its distance is above the threshold.

👀 **Takeaway.** One-to-many recognition needs both a nearest match and a reject option.

### 🟡 Easy Examples

#### E1. Compute embedding distances for verification

**Goal.** Compute distances for several same-person and different-person pairs. We'll build this in **6 steps**.

In [ ]:
pairs_e1 = [(0, 1), (0, 6), (5, 7), (8, 14), (10, 11), (12, 22)]  # choose a mix of same-label and different-label pairs.
distances_e1 = []  # prepare a list for measured pair distances.
labels_e1 = []  # prepare a list where 1 means same identity and 0 means different identity.
for left, right in pairs_e1:  # iterate over selected face-image pairs.
    distance = l2_distance(face_embeddings[left], face_embeddings[right])  # compute the embedding distance for this pair.
    same = int(face_labels[left] == face_labels[right])  # compute the ground-truth verification label.
    distances_e1.append(distance)  # store the distance for plotting and printing.
    labels_e1.append(same)  # store the same/different label.

print("pair | ids | distance | label")  # print a readable table header.

for pair, distance, same in zip(pairs_e1, distances_e1, labels_e1):  # loop over all pair results.
    print(f"{pair} | ({face_labels[pair[0]]}, {face_labels[pair[1]]}) | {distance:.3f} | {'same' if same else 'different'}")  # print one verification row.

In [ ]:
plt.figure(figsize=(7, 3.5))  # create a distance-bar figure.
bar_colors_e1 = [COLORS[2] if same else COLORS[3] for same in labels_e1]  # color same-person and different-person bars differently.
plt.bar(np.arange(len(pairs_e1)), distances_e1, color=bar_colors_e1)  # draw one bar per pair distance.
plt.xticks(np.arange(len(pairs_e1)), [f"{a}-{b}" for a, b in pairs_e1])  # label each bar by its pair indices.
plt.ylabel("L2 embedding distance")  # label the distance axis.
plt.xlabel("image pair indices")  # label the pair axis.
plt.title("E1 verification pair distances")  # title the result plot.
plt.show()  # render the distance bars.

▶ What you'll see: same-identity bars are usually shorter than different-identity bars, which is exactly what a useful embedding space should produce.

#### E2. Choose a verification threshold

**Goal.** Sweep candidate thresholds and pick the one with the best accuracy on labeled pairs. We'll build this in **6 steps**.

In [ ]:
all_pairs_e2 = []  # prepare a list of every unordered pair of synthetic images.
all_distances_e2 = []  # prepare a list of every pair distance.
all_labels_e2 = []  # prepare a list of every pair's same/different label.
for i in range(len(face_embeddings)):  # choose the left image index.
    for j in range(i + 1, len(face_embeddings)):  # choose the right image index without duplicating pairs.
        all_pairs_e2.append((i, j))  # store the pair indices for later inspection.
        all_distances_e2.append(l2_distance(face_embeddings[i], face_embeddings[j]))  # store the pair's L2 distance.
        all_labels_e2.append(int(face_labels[i] == face_labels[j]))  # store 1 for same identity and 0 otherwise.
all_distances_e2 = np.array(all_distances_e2)  # convert distances to an array for vectorized thresholding.
all_labels_e2 = np.array(all_labels_e2)  # convert labels to an array for vectorized accuracy.
threshold_grid_e2 = np.linspace(all_distances_e2.min(), all_distances_e2.max(), 80)  # create candidate thresholds across the observed range.
accuracies_e2 = []  # prepare a list for threshold accuracies.
for threshold in threshold_grid_e2:  # evaluate each candidate threshold.
    predictions = (all_distances_e2 <= threshold).astype(int)  # predict same identity when distance is below threshold.
    accuracies_e2.append(np.mean(predictions == all_labels_e2))  # compute verification accuracy for this threshold.
best_index_e2 = int(np.argmax(accuracies_e2))  # find the index of the best threshold.
best_threshold_e2 = float(threshold_grid_e2[best_index_e2])  # read the best threshold value.
best_accuracy_e2 = float(accuracies_e2[best_index_e2])  # read the best accuracy value.

print(f"Best threshold = {best_threshold_e2:.3f}")  # print the selected threshold.
print(f"Best accuracy = {best_accuracy_e2:.3f}")  # print its validation accuracy.

In [ ]:
plt.figure(figsize=(7, 4))  # create a histogram figure.
plt.hist(all_distances_e2[all_labels_e2 == 1], bins=14, alpha=0.75, color=COLORS[2], label="same identity")  # draw same-person distance distribution.
plt.hist(all_distances_e2[all_labels_e2 == 0], bins=14, alpha=0.55, color=COLORS[3], label="different identity")  # draw different-person distance distribution.
plt.axvline(best_threshold_e2, color="black", linestyle="--", label=f"best threshold {best_threshold_e2:.2f}")  # mark the selected threshold.
plt.xlabel("L2 embedding distance")  # label the histogram axis.
plt.ylabel("number of pairs")  # label the frequency axis.
plt.title("E2 threshold selection from labeled pairs")  # title the threshold plot.
plt.legend()  # show distribution labels and threshold label.
plt.show()  # render the threshold histogram.

▶ What you'll see: the threshold tries to separate short same-person distances from longer different-person distances. Overlap means no threshold can be perfect.

#### E3. Triplet loss on toy embeddings

**Goal.** Evaluate zero-loss and positive-loss triplets geometrically. We'll build this in **5 steps**.

In [ ]:
anchor_e3 = np.array([0.0, 0.0])  # place the anchor at the origin for a simple plot.
positive_e3 = np.array([0.6, 0.2])  # place the positive example close to the anchor.
negative_easy_e3 = np.array([1.8, 0.8])  # place an easy negative far away.
negative_hard_e3 = np.array([0.9, 0.25])  # place a hard negative too close to the anchor.
margin_e3 = 0.5  # set the triplet-loss margin.
def triplet_loss_value(anchor, positive, negative, margin):  # define the scalar triplet loss for one triplet.
    d_ap = l2_distance(anchor, positive)  # compute anchor-positive distance.
    d_an = l2_distance(anchor, negative)  # compute anchor-negative distance.
    loss = max(d_ap - d_an + margin, 0.0)  # apply the hinge formula from CS230.
    return loss, d_ap, d_an  # return loss and component distances for explanation.
loss_easy_e3, d_ap_e3, d_an_easy_e3 = triplet_loss_value(anchor_e3, positive_e3, negative_easy_e3, margin_e3)  # evaluate the easy negative.
loss_hard_e3, _, d_an_hard_e3 = triplet_loss_value(anchor_e3, positive_e3, negative_hard_e3, margin_e3)  # evaluate the hard negative.

print(f"d(A,P) = {d_ap_e3:.3f}")  # print anchor-positive distance.
print(f"easy d(A,N) = {d_an_easy_e3:.3f}, loss = {loss_easy_e3:.3f}")  # print easy-negative result.
print(f"hard d(A,N) = {d_an_hard_e3:.3f}, loss = {loss_hard_e3:.3f}")  # print hard-negative result.

In [ ]:
plt.figure(figsize=(5.5, 5))  # create a square geometry plot.
plt.scatter([anchor_e3[0]], [anchor_e3[1]], s=180, color="black", label="anchor A")  # plot the anchor.
plt.scatter([positive_e3[0]], [positive_e3[1]], s=140, color=COLORS[2], label="positive P")  # plot the positive example.
plt.scatter([negative_easy_e3[0]], [negative_easy_e3[1]], s=140, color=COLORS[0], label="easy negative N")  # plot the easy negative.
plt.scatter([negative_hard_e3[0]], [negative_hard_e3[1]], s=140, color=COLORS[3], label="hard negative N")  # plot the hard negative.
margin_radius_e3 = d_ap_e3 + margin_e3  # compute the required exclusion radius for negatives.
circle_e3 = plt.Circle(anchor_e3, margin_radius_e3, fill=False, linestyle="--", color="gray", label="d(A,P)+margin")  # create a margin circle around the anchor.
plt.gca().add_patch(circle_e3)  # add the margin circle to the axes.
plt.axis("equal")  # keep distances visually faithful.
plt.xlim(-0.4, 2.3)  # set horizontal limits around all points.
plt.ylim(-0.5, 1.5)  # set vertical limits around all points.
plt.xlabel("embedding coordinate 1")  # label horizontal embedding coordinate.
plt.ylabel("embedding coordinate 2")  # label vertical embedding coordinate.
plt.title("E3 triplet margin geometry")  # title the triplet plot.
plt.legend(fontsize=8)  # show point labels and margin label.
plt.show()  # render the triplet geometry.

▶ What you'll see: negatives outside the dashed circle have zero loss; negatives inside the circle violate the margin and create positive triplet loss.

#### E4. Build a mini face-recognition lookup

**Goal.** Compare one query against a gallery of identities and return the nearest neighbor. We'll build this in **6 steps**.

In [ ]:
gallery_indices_e4 = [0, 5, 10, 15, 20]  # choose one enrolled reference image for each of five identities.
query_index_e4 = 7  # choose a query image whose identity should be matched against the gallery.
gallery_embeddings_e4 = face_embeddings[gallery_indices_e4]  # collect gallery embeddings.
gallery_labels_e4 = face_labels[gallery_indices_e4]  # collect gallery identity labels.
query_embedding_e4 = face_embeddings[query_index_e4]  # collect the query embedding.
query_label_e4 = face_labels[query_index_e4]  # collect the query's hidden true identity for evaluation.
distances_e4 = pairwise_l2(query_embedding_e4[None, :], gallery_embeddings_e4).ravel()  # compute query-to-gallery distances.
best_gallery_position_e4 = int(np.argmin(distances_e4))  # find the nearest gallery position.
predicted_identity_e4 = int(gallery_labels_e4[best_gallery_position_e4])  # read the nearest identity label.

print("gallery identity | distance to query")  # print table header.

for identity, distance in zip(gallery_labels_e4, distances_e4):  # loop over gallery distances.
    print(f"{identity:16d} | {distance:.3f}")  # print one gallery candidate row.

print(f"Query true identity = {query_label_e4}")  # print the hidden true query identity.
print(f"Predicted identity = {predicted_identity_e4}")  # print the nearest-neighbor prediction.

In [ ]:
plt.figure(figsize=(7, 3.5))  # create a nearest-neighbor bar plot.
colors_e4 = [COLORS[2] if k == best_gallery_position_e4 else "lightgray" for k in range(len(distances_e4))]  # highlight the nearest gallery bar.
plt.bar(np.arange(len(distances_e4)), distances_e4, color=colors_e4, edgecolor="black")  # draw query-to-gallery distances.
plt.xticks(np.arange(len(distances_e4)), [f"id {identity}" for identity in gallery_labels_e4])  # label each bar by gallery identity.
plt.ylabel("query-to-gallery L2 distance")  # label the distance axis.
plt.xlabel("gallery identity")  # label the gallery axis.
plt.title("E4 one-to-many recognition lookup")  # title the recognition plot.
plt.show()  # render the lookup result.

▶ What you'll see: recognition is one-to-many search. The highlighted nearest bar is the predicted identity; if the true identity is absent or too far, a real system should reject.

#### E5. Neural style transfer first run

**Goal.** Optimize a generated toy tensor to match content and style statistics. We'll build this in **8 steps**.

In [ ]:
generated_e5 = np.clip(content_tensor + RNG.normal(scale=0.08, size=content_tensor.shape), 0.0, 1.0)  # initialize generated tensor near content with small noise.
alpha_e5 = 1.0  # set content weight for the first run.
beta_e5 = 18.0  # set style weight for the first run.
learning_rate_e5 = 1.8  # choose a stable step size for the tiny activation optimization.
steps_e5 = 160  # choose enough steps to visibly reduce the objective on CPU.
snapshots_e5 = []  # prepare a list of generated tensors at selected steps.
losses_e5 = []  # prepare a list of total losses during optimization.
for step in range(steps_e5 + 1):  # run manual gradient descent over the generated tensor.
    total, c_loss, s_loss = total_style_transfer_loss(content_tensor, style_tensor, generated_e5, alpha=alpha_e5, beta=beta_e5)  # evaluate losses at the current tensor.
    losses_e5.append(total)  # store the total loss for a learning curve.
    if step in [0, 40, 80, 160]:  # save snapshots at interpretable milestones.
        snapshots_e5.append((step, generated_e5.copy()))  # store the current generated tensor.
    gradient = style_transfer_grad(content_tensor, style_tensor, generated_e5, alpha=alpha_e5, beta=beta_e5)  # compute manual content-plus-style gradient.
    generated_e5 = np.clip(generated_e5 - learning_rate_e5 * gradient, 0.0, 1.0)  # take a gradient step and clip to displayable values.

print(f"Initial total loss = {losses_e5[0]:.4f}")  # print the starting objective.
print(f"Final total loss = {losses_e5[-1]:.4f}")  # print the ending objective.

In [ ]:
plt.figure(figsize=(6.5, 3.5))  # create a learning-curve figure.
plt.plot(losses_e5, color=COLORS[0])  # draw total objective over optimization steps.
plt.xlabel("gradient step")  # label the horizontal optimization axis.
plt.ylabel("weighted total loss")  # label the loss axis.
plt.title("E5 toy style-transfer optimization curve")  # title the loss curve.
plt.show()  # render the optimization curve.

▶ What you'll see: the total loss falls quickly at first and then flattens as the generated tensor reaches a compromise between content and style.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))  # create a content-style-generated triptych.
show_image_tensor(content_tensor, "content C", ax=axes[0])  # show the content target.
show_image_tensor(style_tensor, "style S", ax=axes[1])  # show the style target.
show_image_tensor(generated_e5, "generated G", ax=axes[2])  # show the final generated tensor.
plt.tight_layout()  # tighten the triptych layout.
plt.show()  # render the first style-transfer result.

▶ What you'll see: the generated tensor keeps the broad content shape while adopting some texture statistics from the style tensor.

In [ ]:
fig, axes = plt.subplots(1, len(snapshots_e5), figsize=(11, 2.8))  # create a row of optimization snapshots.
for ax, (step, snapshot) in zip(axes, snapshots_e5):  # loop over saved generated tensors.
    show_image_tensor(snapshot, f"step {step}", ax=ax)  # display the generated tensor at that step.
plt.tight_layout()  # keep snapshot titles readable.
plt.show()  # render the optimization trajectory.

▶ What you'll see: early snapshots resemble noisy content; later snapshots gradually pick up the style texture while preserving the central structure.

### 🔴 Advanced Examples

#### A1. Hard negative face-verification failure case

**Goal.** Diagnose false accepts and false rejects when same/different distance distributions overlap. We'll build this in **8 steps**.

In [ ]:
hard_embeddings_a1, hard_labels_a1, _ = load_face_data("hard")  # load the intentionally difficult face dataset.
hard_distances_a1 = []  # prepare a list of all hard-case pair distances.
hard_truth_a1 = []  # prepare a list of true same/different labels.
hard_pairs_a1 = []  # prepare a list of pair indices for error inspection.
for i in range(len(hard_embeddings_a1)):  # loop over left image indices.
    for j in range(i + 1, len(hard_embeddings_a1)):  # loop over right image indices without duplicates.
        hard_pairs_a1.append((i, j))  # store pair indices.
        hard_distances_a1.append(l2_distance(hard_embeddings_a1[i], hard_embeddings_a1[j]))  # store pair distance.
        hard_truth_a1.append(int(hard_labels_a1[i] == hard_labels_a1[j]))  # store same/different truth.
hard_distances_a1 = np.array(hard_distances_a1)  # convert hard distances to an array.
hard_truth_a1 = np.array(hard_truth_a1)  # convert hard labels to an array.
threshold_grid_a1 = np.linspace(hard_distances_a1.min(), hard_distances_a1.max(), 100)  # create thresholds to sweep.
accuracies_a1 = []  # prepare accuracy values.
for threshold in threshold_grid_a1:  # evaluate each threshold.
    predictions = (hard_distances_a1 <= threshold).astype(int)  # accept pairs below threshold.
    accuracies_a1.append(np.mean(predictions == hard_truth_a1))  # store hard-case accuracy.
best_threshold_a1 = float(threshold_grid_a1[int(np.argmax(accuracies_a1))])  # choose the best threshold on the hard set.
predictions_a1 = (hard_distances_a1 <= best_threshold_a1).astype(int)  # compute final hard-case predictions.
false_accepts_a1 = np.where((predictions_a1 == 1) & (hard_truth_a1 == 0))[0]  # find different-person pairs incorrectly accepted.
false_rejects_a1 = np.where((predictions_a1 == 0) & (hard_truth_a1 == 1))[0]  # find same-person pairs incorrectly rejected.

print(f"Hard-case best threshold = {best_threshold_a1:.3f}")  # print the threshold.
print(f"False accepts = {len(false_accepts_a1)}")  # print false-accept count.
print(f"False rejects = {len(false_rejects_a1)}")  # print false-reject count.

In [ ]:
plt.figure(figsize=(7, 4))  # create a hard-case histogram.
plt.hist(hard_distances_a1[hard_truth_a1 == 1], bins=16, alpha=0.75, color=COLORS[2], label="same identity")  # plot same-person hard distances.
plt.hist(hard_distances_a1[hard_truth_a1 == 0], bins=16, alpha=0.55, color=COLORS[3], label="different identity")  # plot different-person hard distances.
plt.axvline(best_threshold_a1, color="black", linestyle="--", label="chosen threshold")  # draw the selected threshold.
plt.xlabel("L2 embedding distance")  # label the distance axis.
plt.ylabel("number of pairs")  # label the frequency axis.
plt.title("A1 hard negative overlap")  # title the failure-case plot.
plt.legend()  # show class and threshold labels.
plt.show()  # render the overlap histogram.

▶ What you'll see: overlap near the threshold creates unavoidable mistakes. A look-alike pair can be a false accept, while an occluded same-person pair can be a false reject.

In [ ]:
example_false_accept_a1 = false_accepts_a1[0] if len(false_accepts_a1) else None  # choose one false accept if it exists.
example_false_reject_a1 = false_rejects_a1[0] if len(false_rejects_a1) else None  # choose one false reject if it exists.
for name, index in [("false accept", example_false_accept_a1), ("false reject", example_false_reject_a1)]:  # loop over error types.
    if index is not None:  # print details only when that error type exists.
        pair = hard_pairs_a1[index]  # recover the image-pair indices.
        ids = (hard_labels_a1[pair[0]], hard_labels_a1[pair[1]])  # recover the pair's identity labels.
        distance = hard_distances_a1[index]  # recover the pair distance.
        print(f"{name}: pair {pair}, ids {ids}, distance {distance:.3f}")  # print an interpretable error example.

As an extra A1 diagnostic, sweep the verification threshold to draw a ROC curve for the same accept/reject tradeoff.

In [ ]:
roc_thresholds = np.linspace(all_distances_e2.min() - 0.01, all_distances_e2.max() + 0.01, 120)  # sweep thresholds across and slightly beyond observed distances.
tpr_values = []  # store true positive rates, also called true accept rates.
fpr_values = []  # store false positive rates, also called false accept rates.
for threshold in roc_thresholds:  # evaluate every threshold.
    predicted_same = (all_distances_e2 <= threshold).astype(int)  # accept pairs below the current threshold.
    true_positive = np.sum((predicted_same == 1) & (all_labels_e2 == 1))  # count same-person pairs correctly accepted.
    false_positive = np.sum((predicted_same == 1) & (all_labels_e2 == 0))  # count different-person pairs incorrectly accepted.
    true_negative = np.sum((predicted_same == 0) & (all_labels_e2 == 0))  # count different-person pairs correctly rejected.
    false_negative = np.sum((predicted_same == 0) & (all_labels_e2 == 1))  # count same-person pairs incorrectly rejected.
    tpr = true_positive / max(true_positive + false_negative, 1)  # compute true accept rate safely.
    fpr = false_positive / max(false_positive + true_negative, 1)  # compute false accept rate safely.
    tpr_values.append(tpr)  # store true positive rate.
    fpr_values.append(fpr)  # store false positive rate.
auc_roc = float(np.trapz(np.array(tpr_values)[np.argsort(fpr_values)], np.array(fpr_values)[np.argsort(fpr_values)]))  # approximate ROC area with trapezoids.

print(f"Approximate ROC AUC = {auc_roc:.3f}")  # print ROC area as a threshold-independent summary.

In [ ]:
plt.figure(figsize=(5.5, 5))  # create a square ROC figure.
plt.plot(fpr_values, tpr_values, color=COLORS[0], linewidth=2, label=f"ROC AUC ≈ {auc_roc:.2f}")  # draw the ROC curve.
plt.plot([0, 1], [0, 1], color="gray", linestyle="--", label="random baseline")  # draw random-ranking baseline.
plt.xlabel("false accept rate")  # label x-axis.
plt.ylabel("true accept rate")  # label y-axis.
plt.title("A6 verification ROC from threshold sweep")  # title ROC plot.
plt.legend()  # show curve labels.
plt.axis("square")  # keep ROC geometry square.
plt.show()  # render ROC curve.

▶ What you'll see: moving the threshold changes both true accepts and false accepts. A curve closer to the top-left corner means better verification separation.

#### A2. Triplet mining intuition

**Goal.** Classify negatives as easy, semi-hard, or hard, then see which triplets train the model. We'll build this in **7 steps**.

In [ ]:
anchor_a2 = hard_embeddings_a1[0]  # choose one anchor embedding from the hard dataset.
anchor_label_a2 = hard_labels_a1[0]  # read the anchor identity label.
positive_candidates_a2 = np.where((hard_labels_a1 == anchor_label_a2) & (np.arange(len(hard_labels_a1)) != 0))[0]  # find same-identity positives excluding the anchor.
positive_index_a2 = int(positive_candidates_a2[0])  # choose the first available positive.
positive_a2 = hard_embeddings_a1[positive_index_a2]  # read the positive embedding.
negative_indices_a2 = np.where(hard_labels_a1 != anchor_label_a2)[0]  # find all different-identity negatives.
d_ap_a2 = l2_distance(anchor_a2, positive_a2)  # compute anchor-positive distance.
margin_a2 = 0.35  # set the mining margin.
mining_rows_a2 = []  # prepare rows describing each negative.
for negative_index in negative_indices_a2:  # loop over candidate negatives.
    d_an = l2_distance(anchor_a2, hard_embeddings_a1[negative_index])  # compute anchor-negative distance.
    loss = max(d_ap_a2 - d_an + margin_a2, 0.0)  # compute triplet loss for this negative.
    if d_an < d_ap_a2:  # identify negatives even closer than the positive.
        category = "hard"  # label this as a hard negative.
    elif d_an < d_ap_a2 + margin_a2:  # identify negatives farther than positive but inside the margin.
        category = "semi-hard"  # label this as semi-hard.
    else:  # identify negatives that already satisfy the margin.
        category = "easy"  # label this as easy.
    mining_rows_a2.append((negative_index, d_an, loss, category))  # store mining information.

print(f"Anchor label = {anchor_label_a2}, positive index = {positive_index_a2}, d(A,P) = {d_ap_a2:.3f}")  # summarize anchor-positive geometry.

for row in mining_rows_a2[:10]:  # print the first ten mining rows to keep output readable.
    print(f"negative {row[0]:2d} | d(A,N)={row[1]:.3f} | loss={row[2]:.3f} | {row[3]}")  # show distance, loss, and category.

In [ ]:
projector_a2 = RNG.normal(size=(EMBEDDING_DIM, 2))  # create a random 2-D projection for mining visualization.
projected_a2 = hard_embeddings_a1 @ projector_a2  # project all hard embeddings to two dimensions.
plt.figure(figsize=(6, 5))  # create a mining scatter plot.
plt.scatter(projected_a2[:, 0], projected_a2[:, 1], c=[COLORS[label] for label in hard_labels_a1], s=45, alpha=0.65)  # draw all embeddings colored by identity.
plt.scatter(projected_a2[0, 0], projected_a2[0, 1], s=220, color="black", marker="*", label="anchor")  # highlight the anchor.
plt.scatter(projected_a2[positive_index_a2, 0], projected_a2[positive_index_a2, 1], s=160, color=COLORS[2], edgecolor="black", label="positive")  # highlight the selected positive.
for negative_index, d_an, loss, category in mining_rows_a2[:8]:  # annotate a few candidate negatives.
    marker = "x" if category == "easy" else "D" if category == "semi-hard" else "P"  # choose marker shape by mining category.
    plt.scatter(projected_a2[negative_index, 0], projected_a2[negative_index, 1], s=130, marker=marker, color=COLORS[3], edgecolor="black")  # highlight candidate negative.
plt.title("A2 triplet mining categories in projected space")  # title the mining plot.
plt.xlabel("random projection 1")  # label first projection axis.
plt.ylabel("random projection 2")  # label second projection axis.
plt.legend(fontsize=8)  # show anchor and positive labels.
plt.show()  # render triplet-mining visualization.

▶ What you'll see: easy negatives contribute zero loss, semi-hard negatives are useful because they violate the margin without being closer than the positive, and hard negatives may reveal look-alikes or label noise.

#### A3. Recognition as one-to-many search

**Goal.** Use multiple gallery photos per identity, compute a distance matrix, and inspect top-k retrieval. We'll build this in **8 steps**.

In [ ]:
gallery_mask_a3 = np.isin(face_labels, [0, 1, 2, 3, 4])  # keep all five identities in the gallery.
gallery_embeddings_a3 = face_embeddings[gallery_mask_a3]  # collect gallery embeddings.
gallery_labels_a3 = face_labels[gallery_mask_a3]  # collect gallery labels.
query_indices_a3 = [2, 9, 13, 19, 24]  # choose one query from several identities.
query_embeddings_a3 = face_embeddings[query_indices_a3]  # collect query embeddings.
query_labels_a3 = face_labels[query_indices_a3]  # collect query labels.
distance_matrix_a3 = pairwise_l2(query_embeddings_a3, gallery_embeddings_a3)  # compute query-by-gallery distance matrix.
top_k_a3 = 3  # retrieve the top three nearest gallery photos for each query.
for q_pos, q_label in enumerate(query_labels_a3):  # loop over queries.
    ranking = np.argsort(distance_matrix_a3[q_pos])[:top_k_a3]  # find nearest gallery photo indices.
    retrieved_labels = gallery_labels_a3[ranking]  # read retrieved identity labels.
    retrieved_distances = distance_matrix_a3[q_pos, ranking]  # read retrieved distances.
    print(f"query {q_pos} true id {q_label}: top labels {retrieved_labels.tolist()} with distances {np.round(retrieved_distances, 3).tolist()}")  # print top-k result.

In [ ]:
plt.figure(figsize=(10, 4))  # create a heatmap figure for query-to-gallery distances.
plt.imshow(distance_matrix_a3, aspect="auto", cmap="viridis")  # display smaller distances as one color and larger distances as another.
plt.colorbar(label="L2 distance")  # add a colorbar for distance values.
plt.yticks(np.arange(len(query_indices_a3)), [f"query id {label}" for label in query_labels_a3])  # label rows by query identity.
plt.xticks(np.arange(len(gallery_labels_a3)), [f"{label}" for label in gallery_labels_a3], rotation=90)  # label columns by gallery identity.
plt.xlabel("gallery photo identity label")  # label gallery axis.
plt.ylabel("query")  # label query axis.
plt.title("A3 one-to-many distance matrix")  # title the distance matrix.
plt.tight_layout()  # prevent rotated labels from being clipped.
plt.show()  # render the retrieval heatmap.

▶ What you'll see: each query should have low-distance columns for gallery photos of the same identity. Top-k retrieval is recognition viewed as ranked search.

#### A4. Style/content loss decomposition

**Goal.** Track content loss, style loss, and total loss during toy style-transfer optimization. We'll build this in **10 steps**.

In [ ]:
generated_a4 = np.clip(0.75 * content_tensor + 0.25 * RNG.random(content_tensor.shape), 0.0, 1.0)  # initialize generated tensor as content plus noise.
alpha_a4 = 1.2  # set a content weight that keeps the object visible.
beta_a4 = 24.0  # set a style weight large enough to change texture statistics.
learning_rate_a4 = 1.5  # choose a stable manual gradient step size.
steps_a4 = 220  # run enough steps to see separate loss curves stabilize.
total_curve_a4 = []  # store total loss values.
content_curve_a4 = []  # store unweighted content loss values.
style_curve_a4 = []  # store unweighted style loss values.
snapshots_a4 = []  # store generated tensors at milestones.
for step in range(steps_a4 + 1):  # run gradient descent on the generated tensor.
    total, c_loss, s_loss = total_style_transfer_loss(content_tensor, style_tensor, generated_a4, alpha=alpha_a4, beta=beta_a4)  # compute all losses.
    total_curve_a4.append(total)  # store weighted total loss.
    content_curve_a4.append(c_loss)  # store content loss.
    style_curve_a4.append(s_loss)  # store style loss.
    if step in [0, 50, 110, 220]:  # save interpretable optimization stages.
        snapshots_a4.append((step, generated_a4.copy()))  # store a copy of the generated tensor.
    gradient = style_transfer_grad(content_tensor, style_tensor, generated_a4, alpha=alpha_a4, beta=beta_a4)  # compute manual gradient.
    generated_a4 = np.clip(generated_a4 - learning_rate_a4 * gradient, 0.0, 1.0)  # update generated tensor and clip to display range.

print(f"final content loss = {content_curve_a4[-1]:.4f}")  # print final content loss.
print(f"final style loss = {style_curve_a4[-1]:.4f}")  # print final style loss.
print(f"final total loss = {total_curve_a4[-1]:.4f}")  # print final weighted total loss.

In [ ]:
plt.figure(figsize=(7, 4))  # create a multi-curve loss plot.
plt.plot(total_curve_a4, label="weighted total", color="black")  # draw total weighted objective.
plt.plot(content_curve_a4, label="content loss", color=COLORS[0])  # draw content loss curve.
plt.plot(style_curve_a4, label="style loss", color=COLORS[1])  # draw style loss curve.
plt.xlabel("gradient step")  # label optimization steps.
plt.ylabel("loss value")  # label loss magnitude.
plt.title("A4 content/style loss decomposition")  # title the decomposition plot.
plt.legend()  # show which curve is which.
plt.show()  # render the loss curves.

▶ What you'll see: content and style losses do not always move in perfect sync because the generated tensor is solving a weighted compromise.

In [ ]:
fig, axes = plt.subplots(1, len(snapshots_a4), figsize=(11, 2.8))  # create a row for optimization snapshots.
for ax, (step, snapshot) in zip(axes, snapshots_a4):  # loop over saved snapshots.
    show_image_tensor(snapshot, f"step {step}", ax=ax)  # display each generated tensor.
plt.tight_layout()  # keep snapshot row readable.
plt.show()  # render A4 snapshots.

▶ What you'll see: the generated tensor gradually changes texture while retaining the high-level content layout.

#### A5. Style-transfer tradeoff and edge case

**Goal.** Vary $\alpha/\beta$ and compare a smooth style with a noisy style artifact. We'll build this in **10 steps**.

In [ ]:
styles_a5 = {"swirl": make_toy_image("swirl", size=16), "noise": make_toy_image("noise", size=16)}  # create two style targets, one structured and one noisy.
settings_a5 = [(3.0, 6.0, "content-heavy"), (1.0, 18.0, "balanced"), (0.35, 45.0, "style-heavy")]  # define three content/style tradeoffs.
results_a5 = {}  # prepare a dictionary to hold final generated tensors and losses.
for style_name, style_target in styles_a5.items():  # loop over structured and noisy styles.
    results_a5[style_name] = []  # create a result list for this style.
    for alpha, beta, label in settings_a5:  # loop over tradeoff settings.
        generated = np.clip(content_tensor + RNG.normal(scale=0.06, size=content_tensor.shape), 0.0, 1.0)  # initialize generated tensor near content.
        for step in range(150):  # run a short optimization for this tradeoff.
            gradient = style_transfer_grad(content_tensor, style_target, generated, alpha=alpha, beta=beta)  # compute gradient for this style and tradeoff.
            generated = np.clip(generated - 0.25 * gradient, 0.0, 1.0)  # update generated tensor with a conservative fixed learning rate.
        total, c_loss, s_loss = total_style_transfer_loss(content_tensor, style_target, generated, alpha=alpha, beta=beta)  # evaluate final losses.
        results_a5[style_name].append((label, generated, total, c_loss, s_loss))  # store label, image, and losses.
        print(f"{style_name:5s} | {label:13s} | total={total:.4f} | content={c_loss:.4f} | style={s_loss:.4f}")  # print a compact result row.

In [ ]:
fig, axes = plt.subplots(len(styles_a5), len(settings_a5) + 1, figsize=(12, 5.5))  # create a grid comparing styles and tradeoffs.
for row, (style_name, style_target) in enumerate(styles_a5.items()):  # loop over style rows.
    show_image_tensor(style_target, f"style: {style_name}", ax=axes[row, 0])  # show the style target at the left.
    for col, (label, generated, total, c_loss, s_loss) in enumerate(results_a5[style_name], start=1):  # loop over generated results.
        show_image_tensor(generated, label, ax=axes[row, col])  # show one generated tensor for this tradeoff.
plt.tight_layout()  # tighten the comparison grid.
plt.show()  # render the tradeoff grid.

▶ What you'll see: content-heavy settings preserve the blob and edge, style-heavy settings exaggerate texture, and the noisy style can inject speckled artifacts even when the loss improves.

### Interactive Experiment

Move the distance threshold to see accept/reject decisions and accuracy update live. In Colab, the slider is interactive; outside Colab, the fallback runs once at the default threshold.

In [ ]:
def threshold_experiment(threshold=best_threshold_e2):  # define the live threshold experiment callback.
    predictions = (all_distances_e2 <= threshold).astype(int)  # accept all pairs whose distance is below the slider threshold.
    accuracy = float(np.mean(predictions == all_labels_e2))  # compute current verification accuracy.
    false_accept_rate = float(np.mean((predictions == 1) & (all_labels_e2 == 0)))  # compute fraction of all pairs that are false accepts.
    false_reject_rate = float(np.mean((predictions == 0) & (all_labels_e2 == 1)))  # compute fraction of all pairs that are false rejects.
    plt.figure(figsize=(7, 4))  # create a fresh plot for the current slider value.
    plt.hist(all_distances_e2[all_labels_e2 == 1], bins=14, alpha=0.75, color=COLORS[2], label="same identity")  # plot same-person distances.
    plt.hist(all_distances_e2[all_labels_e2 == 0], bins=14, alpha=0.55, color=COLORS[3], label="different identity")  # plot different-person distances.
    plt.axvline(threshold, color="black", linestyle="--", linewidth=2, label=f"threshold {threshold:.2f}")  # draw the interactive threshold.
    plt.xlabel("L2 embedding distance")  # label distance axis.
    plt.ylabel("number of pairs")  # label count axis.
    plt.title(f"accuracy={accuracy:.3f}, false accepts={false_accept_rate:.3f}, false rejects={false_reject_rate:.3f}")  # summarize current metrics in the title.
    plt.legend()  # show histogram and threshold labels.
    plt.show()  # render the interactive plot.
    print("Accepted pairs are to the left of the threshold; rejected pairs are to the right.")  # explain the decision rule.
    print(f"Current threshold: {threshold:.3f}")  # print current threshold.
    print(f"Current accuracy: {accuracy:.3f}")  # print current accuracy.
    return accuracy  # return accuracy so notebook users see a final scalar.

interact(threshold_experiment, threshold=FloatSlider(value=best_threshold_e2, min=float(all_distances_e2.min()), max=float(all_distances_e2.max()), step=0.01, description="threshold"))  # launch the slider experiment.

▶ What you'll see: lowering the threshold reduces false accepts but increases false rejects; raising it does the opposite. Verification is not just a model problem—it is also an operating-point choice.